# 뉴스 사실검증 데이터 EDA

기사 전처리는 `src/news_preprocessor.py`, 주장 정규화는 `src/claim_normalizer.py`를 사용하며, 이 노트북은 분석과 주장 추출에 집중합니다.


In [2]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 데이터 불러오기

In [8]:
from pathlib import Path

csv_path = Path("/content/drive/MyDrive/멋사/data/AI_기반_뉴스_사실검증_시스템_프로젝트_데이터.csv")
df_raw = pd.read_csv(csv_path, encoding="utf-8")

## 2. 기사 전처리 모듈 적용

노트북에는 전처리 규칙을 두지 않습니다. 기사 본문 정제는 `src/news_preprocessor.py`, 주장 시점·단위 정규화는 `src/claim_normalizer.py`를 사용합니다.


In [6]:
import sys

project_candidates = [Path.cwd(), csv_path.parent.parent]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "src" / "news_preprocessor.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트의 src 폴더를 찾을 수 없습니다. Google Drive에 src 폴더를 업로드했는지 확인하세요."
    )

project_root_text = str(PROJECT_ROOT.resolve())
if project_root_text not in sys.path:
    sys.path.insert(0, project_root_text)

from src.news_preprocessor import preprocess, summarize
from src.claim_normalizer import resolve_relative_time, convert_value

BODY_COLUMN = "기사 본문 전체"
CLEAN_BODY_COLUMN = "본문_정제"


In [ ]:
df = preprocess(df_raw)
print(summarize(df))
display(df[["기사제목", BODY_COLUMN, CLEAN_BODY_COLUMN]].head(3))


NameError: name 'preprocess' is not defined

## 4. 전처리 결과 저장


In [ ]:
output_path = csv_path.with_name(f"{csv_path.stem}_본문전처리.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")


저장 완료: /content/drive/MyDrive/멋사/data/C:\Users\82105\Desktop\likelion5\data\AI_기반_뉴스_사실검증_시스템_프로젝트_데이터_본문전처리.csv


## 4-1. 저장된 전처리 데이터 불러오기

In [13]:
BODY_COLUMN = "기사 본문 전체"
CLEAN_BODY_COLUMN = "본문_정제"

df = pd.read_csv("/content/drive/MyDrive/멋사/data/AI_기반_뉴스_사실검증_시스템_프로젝트_데이터_본문전처리.csv", encoding="utf-8")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2706 entries, 0 to 2705
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   기사제목       2706 non-null   object
 1   작성일        2706 non-null   object
 2   URL        2706 non-null   object
 3   기사 본문 전체   2705 non-null   object
 4   검색 구분 레이블  2706 non-null   bool  
 5   본문_정제      2701 non-null   object
 6   작성자        2643 non-null   object
 7   섹션         2706 non-null   object
 8   헤더_제거됨     2706 non-null   bool  
 9   푸터_제거됨     2706 non-null   bool  
 10  포맷         2706 non-null   object
 11  위젯잔재_의심    2706 non-null   bool  
 12  본문_상태      9 non-null      object
 13  본문_길이      2706 non-null   int64 
 14  문장수        2706 non-null   int64 
dtypes: bool(4), int64(2), object(9)
memory usage: 243.2+ KB


## 6. NCP CLOVA Studio로 기사 주장 추출

전처리된 `기사 본문 전처리`에서 검증 가능한 주장과 수치 정보를 추출합니다. `.env`의 `NCP_CLOVASTUDIO_API_KEY=` 뒤에 키를 입력한 후 실행하세요. 기본값은 비용 보호를 위해 1건 테스트 및 최대 10건 배치입니다.

In [5]:
import json
import os
import time
import uuid
from pathlib import Path

import requests

### 6-A. API 키 설정 및 프롬프트

In [9]:
def load_env_file(path: Path) -> None:
    if not path.is_file():
        return
    for raw_line in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))

# 로컬 프로젝트와 Google Drive 데이터 폴더 양쪽을 지원합니다.
for env_path in [Path.cwd() / ".env", csv_path.parent / ".env"]:
    load_env_file(env_path)

NCP_API_KEY = os.getenv("NCP_CLOVASTUDIO_API_KEY", "").strip()
NCP_MODEL = "HCX-007"
NCP_API_URL = f"https://clovastudio.stream.ntruss.com/v3/chat-completions/{NCP_MODEL}"
MAX_CHARS_PER_CHUNK = 40_000
CHUNK_OVERLAP = 500
REQUEST_INTERVAL_SECONDS = 0.5

print("NCP API 키를 불러왔습니다." if NCP_API_KEY else "NCP API 키가 없습니다. .env에 입력한 후 다시 실행하세요.")

NCP API 키를 불러왔습니다.


In [10]:
CLAIM_SCHEMA = {
    "type": "object",
    "properties": {
        "claims": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "claim_text": {"type": "string", "description": "기사에 명시된 완결된 주장"},
                    "claim_type": {"type": "string", "enum": ["numeric", "non_numeric"]},
                    "subject": {"type": "string"},
                    "predicate": {"type": "string"},
                    "object": {"type": "string"},
                    "numeric_values": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "raw_value": {"type": "string"},
                                "normalized_value": {"type": "string"},
                                "unit": {"type": "string"},
                                "context": {"type": "string"}
                            },
                            "required": ["raw_value", "normalized_value", "unit", "context"]
                        }
                    },
                    "evidence_quote": {"type": "string", "description": "본문의 짧은 직접 근거 문구"}
                },
                "required": ["claim_text", "claim_type", "subject", "predicate", "object", "numeric_values", "evidence_quote"]
            }
        }
    },
    "required": ["claims"]
}

SYSTEM_PROMPT = """
당신은 뉴스 기사에서 통계 자료로 검증 가능한 Claim을 추출하는 분석가입니다.

목표:
뉴스 문장에서 통계표와 비교할 수 있는 독립적인 주장을 추출하고
지정된 구조로 변환합니다.

추출 대상:
- 특정 시점의 수치
- 비율 및 구성비
- 증가율과 감소율
- 두 대상 또는 두 시점의 비교
- 순위
- 최고·최저
- 일정 기간의 추세
- 이상·이하·초과·미만 등의 범위 주장

제외 대상:
- 주관적 평가
- 전망과 희망
- 수사적 표현
- 통계로 검증할 수 없는 기업 관계자의 개인 경험
- 단순 날짜, 주소, 인원 소개
- 의미가 불분명한 숫자

규칙:
1. 한 문장에 여러 Claim이 있으면 각각 분리합니다.
2. 원문에 없는 숫자나 단위를 생성하지 않습니다.
3. claim_text는 앞 문맥 없이도 이해 가능한 완결된 문장으로 작성합니다.
4. subject, predicate, object는 원문의 의미를 유지해 채웁니다.
5. '약', '가량', '이상', '넘는' 등의 한정 표현을 보존합니다.
6. 수치가 있는 주장은 numeric, 수치 없이 순위·증감·최고·최저를 말하면 non_numeric으로 분류합니다.
7. numeric_values에는 원문에 실제 등장한 값만 넣고, 수치가 없으면 빈 배열을 사용합니다.
8. evidence_quote에는 해당 주장을 직접 뒷받침하는 원문 일부를 넣습니다.
9. 추출 대상에 해당하는 주장을 빠뜨리지 말되, 근거가 없으면 생성하지 않습니다.

"""

def split_article(text: str, max_chars: int = MAX_CHARS_PER_CHUNK, overlap: int = CHUNK_OVERLAP) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            boundary = max(text.rfind(". ", start, end), text.rfind("다. ", start, end))
            if boundary > start + max_chars // 2:
                end = boundary + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

def call_clova_claim_api(article_text: str, max_retries: int = 3) -> tuple[dict, dict]:
    if not NCP_API_KEY:
        raise RuntimeError("NCP_CLOVASTUDIO_API_KEY가 설정되지 않았습니다.")
    headers = {
        "Authorization": f"Bearer {NCP_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"다음 기사에서 주장을 추출하세요.\n\n{article_text}"},
        ],
        "topP": 0.8, "topK": 0, "temperature": 0.1,
        "repetitionPenalty": 1.1, "maxCompletionTokens": 4096,
        "thinking": {"effort": "none"},
        "responseFormat": {"type": "json", "schema": CLAIM_SCHEMA},
    }
    for attempt in range(max_retries):
        response = requests.post(NCP_API_URL, headers=headers, json=body, timeout=180)
        if response.status_code == 429 or response.status_code >= 500:
            if attempt + 1 < max_retries:
                time.sleep(2 ** attempt)
                continue
        response.raise_for_status()
        payload = response.json()
        result = payload.get("result", payload)
        content = result.get("message", {}).get("content", "")
        if not content:
            raise RuntimeError(f"CLOVA 응답에 content가 없습니다: {payload}")
        return json.loads(content), result.get("usage", {})
    raise RuntimeError("CLOVA API 재시도 횟수를 초과했습니다.")

def extract_claims_from_article(article_text: str) -> tuple[list[dict], dict]:
    unique, total_usage = {}, {"promptTokens": 0, "completionTokens": 0, "totalTokens": 0}
    for chunk_number, chunk in enumerate(split_article(article_text), 1):
        parsed, usage = call_clova_claim_api(chunk)
        for claim in parsed.get("claims", []):
            claim["chunk_number"] = chunk_number
            unique.setdefault(claim.get("claim_text", "").strip(), claim)
        for key in total_usage:
            total_usage[key] += int(usage.get(key, 0) or 0)
        time.sleep(REQUEST_INTERVAL_SECONDS)
    unique.pop("", None)
    return list(unique.values()), total_usage

### 6-A. 한 기사로 테스트

In [14]:
# 첫 행 고정 대신 통계 검증에 적합한 표현이 많은 기사를 선택합니다.
TEST_ARTICLE_INDEX = None  # 특정 df 인덱스를 시험하려면 정수로 지정
TEST_SIGNAL_PATTERN = r"%|퍼센트|증가|감소|늘었|줄었|최고|최저|평균|배|명|건|원|억|만"

test_rows = df[df[CLEAN_BODY_COLUMN].notna()].copy()
if TEST_ARTICLE_INDEX is None and not test_rows.empty:
    test_rows["_claim_signal_score"] = test_rows[CLEAN_BODY_COLUMN].astype(str).str.count(TEST_SIGNAL_PATTERN)
    test_row = test_rows.sort_values("_claim_signal_score", ascending=False).iloc[0]
elif TEST_ARTICLE_INDEX is not None:
    test_row = df.loc[TEST_ARTICLE_INDEX]
if not NCP_API_KEY:
    print("API 키를 입력한 뒤 설정 셀부터 다시 실행하세요.")
elif test_rows.empty:
    print("전처리된 본문이 없습니다.")
else:
    print(f"테스트 기사: {test_row.get('기사제목', '(제목 없음)')}")
    print(f"df 인덱스: {test_row.name} / 본문 길이: {len(str(test_row[CLEAN_BODY_COLUMN])):,}자")
    test_claims, test_usage = extract_claims_from_article(test_row[CLEAN_BODY_COLUMN])
    print(f"추출 주장: {len(test_claims)}개 / 토큰 사용량: {test_usage}")
    if test_claims:
        display(pd.json_normalize(test_claims))
    else:
        print("모델이 추출 가능한 통계 주장을 찾지 못했습니다. TEST_ARTICLE_INDEX로 다른 기사를 지정해 보세요.")
        print("본문 미리보기:", str(test_row[CLEAN_BODY_COLUMN])[:500])

추출 주장: 0개 / 토큰 사용량: {'promptTokens': 946, 'completionTokens': 6, 'totalTokens': 952}


""


In [16]:
test_row

,0
기사제목,"9명 가족 잃고, 하염없이 기다리던 푸딩이... 동물단체가 구조"
작성일,2025-01-01
URL,https://www.chosun.com/national/national_gener...
기사 본문 전체,"9명 가족 잃고, 하염없이 기다리던 푸딩이... 동물단체가 구조 이혜진 기자 입력 ..."
검색 구분 레이블,False
본문_정제,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다...
작성자,이혜진 기자
섹션,national
헤더_제거됨,True
푸터_제거됨,False


In [18]:
test_row = test_rows.iloc[1]
test_claims, test_usage = extract_claims_from_article(test_row[CLEAN_BODY_COLUMN])
print(f"추출 주장: {len(test_claims)}개 / 토큰 사용량: {test_usage}")
display(pd.json_normalize(test_claims))

추출 주장: 2개 / 토큰 사용량: {'promptTokens': 801, 'completionTokens': 417, 'totalTokens': 1218}


,claim_text,claim_type,subject,predicate,object,numeric_values,evidence_quote,chunk_number
0,"폴크스바겐과 아우디, 세아트, 스코다 등 폴크스바겐그룹 전기차 80만대에서 수집한 ...",numeric,"폴크스바겐과 아우디, 세아트, 스코다 등 폴크스바겐그룹 전기차",수집한 데이터 양,수 테라바이트,"[{'raw_value': '80만대', 'normalized_value': '80...","폴크스바겐과 아우디, 세아트, 스코다 등 폴크스바겐그룹 전기차 80만대에서 수집한 ...",1
1,대상 차량은 독일 30만대 등 대부분 유럽 국가에 집중됐다.,numeric,대상 차량,위치,독일 30만대 등 대부분 유럽 국가,"[{'raw_value': '30만대', 'normalized_value': '30...",대상 차량은 독일 30만대 등 대부분 유럽 국가에 집중됐다.,1


In [22]:
test_row['본문_정제']

'전기차 전환에 뒤처지며 창사 87년 만에 처음으로 독일 자국 내 공장 폐쇄에 나선 폴크스바겐이 이번엔 전기차 약 80만대의 운행 데이터와 소유주 정보를 온라인에서 부실 관리했다는 논란에 휩싸였다. 전기차에 이어 미래 자동차의 핵심인 소프트웨어 관리에서도 약점을 드러냈다는 지적이 나온다. 다만 회사 측은 "차량 소유주의 개인 정보가 외부로 유출된 흔적은 나오지 않았다"고 밝혔다. 지난달 31일 독일 슈피겔 등 외신에 따르면, 폴크스바겐과 아우디, 세아트, 스코다 등 폴크스바겐그룹 전기차 80만대에서 수집한 수 테라바이트 규모 데이터가 지난 몇 달간 암호화되지 않은 채로 아마존 클라우드에 방치됐던 것으로 나타났다. 전기차가 어디에서 어디로 향했는지, 얼마나 머물렀는지와 같은 주행 데이터를 비롯해 운전자의 이메일, 전화번호, 주소 등의 정보다. 차량 위치 정보의 경우 10cm 단위까지 구별될 정도로 정확도가 높았다고 한다. 대상 차량은 독일 30만대 등 대부분 유럽 국가에 집중됐다. 이번 문제는 폴크스바겐 내부 고발자가 슈피겔과 독일 보안 해커 단체인 CCC에 제보하면서 알려졌고, 회사는 뒤늦게 해당 정보에 대한 외부 접근을 막는 조치를 취했다. 폴크스바겐 측은 "우리가 아는 한 해커 단체 외에 시스템에 접근한 경우가 없고, 개인 비밀번호나 결제 정보 등은 해당 없다"고 했다.'

In [23]:
test_claims

[{'claim_text': '폴크스바겐과 아우디, 세아트, 스코다 등 폴크스바겐그룹 전기차 80만대에서 수집한 수 테라바이트 규모의 데이터',
  'claim_type': 'numeric',
  'subject': '폴크스바겐과 아우디, 세아트, 스코다 등 폴크스바겐그룹 전기차',
  'predicate': '수집한 데이터 양',
  'object': '수 테라바이트',
  'numeric_values': [{'raw_value': '80만대',
    'normalized_value': '800,000',
    'unit': '',
    'context': '폴크스바겐 그룹 전기차 대수'},
   {'raw_value': '수 테라바이트',
    'normalized_value': '수 테라바이트',
    'unit': '데이터 크기',
    'context': ''}],
  'evidence_quote': '폴크스바겐과 아우디, 세아트, 스코다 등 폴크스바겐그룹 전기차 80만대에서 수집한 수 테라바이트 규모 데이터가 지난 몇 달간 암호화되지 않은 채로 아마존 클라우드에 방치됐던 것으로 나타났다.',
  'chunk_number': 1},
 {'claim_text': '대상 차량은 독일 30만대 등 대부분 유럽 국가에 집중됐다.',
  'claim_type': 'numeric',
  'subject': '대상 차량',
  'predicate': '위치',
  'object': '독일 30만대 등 대부분 유럽 국가',
  'numeric_values': [{'raw_value': '30만대',
    'normalized_value': '300,000',
    'unit': '',
    'context': '독일의 대상 차량 대수'}],
  'evidence_quote': '대상 차량은 독일 30만대 등 대부분 유럽 국가에 집중됐다.',
  'chunk_number': 1}]

### 6-A. 배치 추출 및 체크포인트 저장

`MAX_ARTICLES`를 `None`으로 바꾸면 전체 기사를 처리합니다. 먼저 소량으로 비용과 결과를 확인하세요.

In [ ]:
MAX_ARTICLES = 10  # 전체 실행은 None
CLAIMS_OUTPUT_PATH = csv_path.with_name(f"{csv_path.stem}_주장추출.csv")
CHECKPOINT_PATH = csv_path.with_name(f"{csv_path.stem}_주장추출_checkpoint.jsonl")

def load_completed_urls(path: Path) -> set[str]:
    if not path.is_file():
        return set()
    completed = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            record = json.loads(line)
            if record.get("status") == "success":
                completed.add(record.get("url", ""))
        except json.JSONDecodeError:
            continue
    return completed

if not NCP_API_KEY:
    print("API 키를 입력한 뒤 이 셀을 실행하세요.")
else:
    completed_urls = load_completed_urls(CHECKPOINT_PATH)
    targets = df[df[CLEAN_BODY_COLUMN].notna() & ~df["URL"].isin(completed_urls)]
    if MAX_ARTICLES is not None:
        targets = targets.head(MAX_ARTICLES)
    print(f"이번 실행 대상: {len(targets):,}건 / 기존 완료: {len(completed_urls):,}건")

    with CHECKPOINT_PATH.open("a", encoding="utf-8") as checkpoint:
        for sequence, (_, row) in enumerate(targets.iterrows(), 1):
            record = {"url": row["URL"], "title": row["기사제목"]}
            try:
                claims, usage = extract_claims_from_article(row[CLEAN_BODY_COLUMN])
                record.update({"status": "success", "claims": claims, "usage": usage})
                print(f"[{sequence}/{len(targets)}] 성공: {len(claims)}개 - {row['기사제목'][:40]}")
            except Exception as error:
                record.update({"status": "failed", "error": str(error), "claims": []})
                print(f"[{sequence}/{len(targets)}] 실패: {error}")
            checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
            checkpoint.flush()

    flat_rows = []
    for line in CHECKPOINT_PATH.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        if record.get("status") != "success":
            continue
        for claim_number, claim in enumerate(record.get("claims", []), 1):
            flat_rows.append({
                "URL": record["url"], "기사제목": record["title"],
                "claim_number": claim_number, **claim,
                "numeric_values_json": json.dumps(claim.get("numeric_values", []), ensure_ascii=False),
            })
    claims_df = pd.DataFrame(flat_rows)
    if "numeric_values" in claims_df.columns:
        claims_df = claims_df.drop(columns=["numeric_values"])
    claims_df.to_csv(CLAIMS_OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {CLAIMS_OUTPUT_PATH} ({len(claims_df):,}개 주장)")

### 6-B. 6-A 동일 구조의 few-shot 실험

6-A와 공정하게 비교하기 위해 입력 데이터, `CLAIM_SCHEMA`, 청킹, HCX 파라미터, 중복 제거, 체크포인트 및 CSV 저장 구조를 모두 동일하게 유지합니다. 유일한 차이는 `candidate_labeling_pilot_relaxed_team2.csv`의 실제 라벨 문장을 user/assistant 예시로 프롬프트에 삽입한다는 점입니다.


#### 6-B-1. 라벨 데이터에서 few-shot 예시 생성

양성 2건과 음성 2건을 고정 정렬해 선택합니다. 예시 답변도 6-A의 `CLAIM_SCHEMA`와 같은 형태로 만들며, 직접 작성한 예시 문장은 사용하지 않습니다.


In [86]:
def find_6b_label_file(filename="candidate_labeling_pilot_relaxed_team2.csv"):
    candidates = [
        PROJECT_ROOT / "data" / filename,
        csv_path.parent / filename,
        Path("/content/drive/MyDrive/news_verification/data") / filename,
        Path("/content/drive/MyDrive/news_verification/data/통계표") / filename,
        Path("/content") / filename,
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError("라벨 파일을 찾지 못했습니다:\n" + "\n".join(map(str, candidates)))


CANDIDATE_LABEL_6B_PATH = find_6b_label_file()
candidate_6b_df = pd.read_csv(CANDIDATE_LABEL_6B_PATH, encoding="utf-8-sig")


def _is_true(value):
    return value is True or str(value).strip().lower() in {"true", "1", "yes"}


def _split_gold(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    return [item.strip() for item in str(value).split(";") if item.strip()]


def _candidate_row_to_claim(row):
    values = _split_gold(row.get("gold_value")) or _split_gold(row.get("value_list"))
    units = _split_gold(row.get("gold_unit")) or _split_gold(row.get("unit_list"))
    numeric_values = []
    for index, raw_value in enumerate(values):
        unit = units[index] if index < len(units) else (units[-1] if units else "")
        numeric_values.append({
            "raw_value": raw_value,
            "normalized_value": raw_value,
            "unit": unit,
            "context": str(row.get("gold_indicator_raw") or row.get("claim_text") or ""),
        })
    subject = str(row.get("gold_population") or row.get("gold_indicator_raw") or "")
    predicate = str(row.get("change_type") or row.get("gold_claim_class") or "")
    return {
        "claim_text": str(row["claim_text"]),
        "claim_type": "numeric" if numeric_values else "non_numeric",
        "subject": subject,
        "predicate": predicate,
        "object": str(row.get("gold_indicator_raw") or ""),
        "numeric_values": numeric_values,
        "evidence_quote": str(row.get("gold_source_evidence_quote") or row["claim_text"]),
    }


def build_fewshot_examples_6b(label_df, positive_count=2, negative_count=2):
    positive_mask = label_df["gold_is_aggregate_claim"].map(_is_true)
    positives = label_df[positive_mask].sort_values("sample_id").head(positive_count)
    negatives = label_df[~positive_mask].sort_values("sample_id").head(negative_count)
    if len(positives) < positive_count or len(negatives) < negative_count:
        raise ValueError("few-shot 양성 또는 음성 예시가 부족합니다.")
    examples = []
    for _, row in pd.concat([positives, negatives]).iterrows():
        claims = [_candidate_row_to_claim(row)] if _is_true(row["gold_is_aggregate_claim"]) else []
        examples.append({
            "sample_id": str(row["sample_id"]),
            "user": f"다음 기사에서 주장을 추출하세요.\n\n{row['claim_text']}",
            "assistant": {"claims": claims},
        })
    return examples


FEWSHOT_EXAMPLES_6B = build_fewshot_examples_6b(candidate_6b_df)
FEWSHOT_EXAMPLE_IDS_6B = [example["sample_id"] for example in FEWSHOT_EXAMPLES_6B]
print("6-B few-shot 예시:", [(x["sample_id"], len(x["assistant"]["claims"])) for x in FEWSHOT_EXAMPLES_6B])


6-B few-shot 예시: [('C001', 1), ('C004', 1), ('C002', 0), ('C003', 0)]


#### 6-B-2. 6-A 프롬프트의 few-shot 변형

`SYSTEM_PROMPT`와 현재 기사 요청은 6-A와 같습니다. 그 사이에 라벨 데이터로 만든 예시 대화만 추가합니다.


In [ ]:
def build_fewshot_messages_6b(article_text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for example in FEWSHOT_EXAMPLES_6B:
        messages.extend([
            {"role": "user", "content": example["user"]},
            {"role": "assistant", "content": json.dumps(example["assistant"], ensure_ascii=False)},
        ])
    messages.append({
        "role": "user",
        "content": f"다음 기사에서 주장을 추출하세요.\n\n{article_text}",
    })
    return messages


def call_clova_claim_api_fewshot(article_text: str, max_retries: int = 3) -> tuple[dict, dict]:
    if not NCP_API_KEY:
        raise RuntimeError("NCP_CLOVASTUDIO_API_KEY가 설정되지 않았습니다.")
    headers = {
        "Authorization": f"Bearer {NCP_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    body = {
        "messages": build_fewshot_messages_6b(article_text),
        "topP": 0.8, "topK": 0, "temperature": 0.1,
        "repetitionPenalty": 1.1, "maxCompletionTokens": 4096,
        "thinking": {"effort": "none"},
        "responseFormat": {"type": "json", "schema": CLAIM_SCHEMA},
    }
    for attempt in range(max_retries):
        response = requests.post(NCP_API_URL, headers=headers, json=body, timeout=180)
        if response.status_code == 429 or response.status_code >= 500:
            if attempt + 1 < max_retries:
                time.sleep(2 ** attempt)
                continue
        response.raise_for_status()
        payload = response.json()
        result = payload.get("result", payload)
        content = result.get("message", {}).get("content", "")
        if not content:
            raise RuntimeError(f"CLOVA 응답에 content가 없습니다: {payload}")
        return json.loads(content), result.get("usage", {})
    raise RuntimeError("CLOVA API 재시도 횟수를 초과했습니다.")


def extract_claims_from_article_fewshot(article_text: str) -> tuple[list[dict], dict]:
    unique, total_usage = {}, {"promptTokens": 0, "completionTokens": 0, "totalTokens": 0}
    for chunk_number, chunk in enumerate(split_article(article_text), 1):
        parsed, usage = call_clova_claim_api_fewshot(chunk)
        for claim in parsed.get("claims", []):
            claim["chunk_number"] = chunk_number
            unique.setdefault(claim.get("claim_text", "").strip(), claim)
        for key in total_usage:
            total_usage[key] += int(usage.get(key, 0) or 0)
        time.sleep(REQUEST_INTERVAL_SECONDS)
    unique.pop("", None)
    return list(unique.values()), total_usage


print("6-B message roles:", [message["role"] for message in build_fewshot_messages_6b("평가 기사")])


6-B message roles: ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user']


#### 6-B-3. 한 기사로 테스트

6-A의 테스트 셀과 동일한 행을 사용하므로 출력 차이는 few-shot 예시의 영향으로 해석할 수 있습니다.


In [88]:
test_rows_6b = df[df[CLEAN_BODY_COLUMN].notna()]
if not NCP_API_KEY:
    print("API 키를 입력한 뒤 설정 셀부터 다시 실행하세요.")
elif test_rows_6b.empty:
    print("전처리된 본문이 없습니다.")
else:
    test_row_6b = test_rows_6b.iloc[0]
    test_claims_6b, test_usage_6b = extract_claims_from_article_fewshot(test_row_6b[CLEAN_BODY_COLUMN])
    print(f"추출 주장: {len(test_claims_6b)}개 / 토큰 사용량: {test_usage_6b}")
    display(pd.json_normalize(test_claims_6b))


추출 주장: 0개 / 토큰 사용량: {'promptTokens': 1655, 'completionTokens': 6, 'totalTokens': 1661}


""


#### 6-B-4. 배치 추출 및 체크포인트 저장

6-A와 같은 배치 로직이며 산출물 파일명에 `_fewshot`을 붙여 zero-shot 결과를 덮어쓰지 않습니다. `MAX_ARTICLES_6B=None`이면 전체 기사를 처리합니다.


In [ ]:
MAX_ARTICLES_6B = 10  # 전체 실행은 None
CLAIMS_OUTPUT_PATH_6B = csv_path.with_name(f"{csv_path.stem}_주장추출_fewshot.csv")
CHECKPOINT_PATH_6B = csv_path.with_name(f"{csv_path.stem}_주장추출_fewshot_checkpoint.jsonl")

if not NCP_API_KEY:
    print("API 키를 입력한 뒤 이 셀을 실행하세요.")
else:
    completed_urls_6b = load_completed_urls(CHECKPOINT_PATH_6B)
    targets_6b = df[df[CLEAN_BODY_COLUMN].notna() & ~df["URL"].isin(completed_urls_6b)]
    if MAX_ARTICLES_6B is not None:
        targets_6b = targets_6b.head(MAX_ARTICLES_6B)
    print(f"이번 실행 대상: {len(targets_6b):,}건 / 기존 완료: {len(completed_urls_6b):,}건")

    with CHECKPOINT_PATH_6B.open("a", encoding="utf-8") as checkpoint:
        for sequence, (_, row) in enumerate(targets_6b.iterrows(), 1):
            record = {"url": row["URL"], "title": row["기사제목"]}
            try:
                claims, usage = extract_claims_from_article_fewshot(row[CLEAN_BODY_COLUMN])
                record.update({"status": "success", "claims": claims, "usage": usage})
                print(f"[{sequence}/{len(targets_6b)}] 성공: {len(claims)}개 - {row['기사제목'][:40]}")
            except Exception as error:
                record.update({"status": "failed", "error": str(error), "claims": []})
                print(f"[{sequence}/{len(targets_6b)}] 실패: {error}")
            checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
            checkpoint.flush()

    flat_rows_6b = []
    for line in CHECKPOINT_PATH_6B.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        if record.get("status") != "success":
            continue
        for claim_number, claim in enumerate(record.get("claims", []), 1):
            flat_rows_6b.append({
                "URL": record["url"], "기사제목": record["title"],
                "claim_number": claim_number, **claim,
                "numeric_values_json": json.dumps(claim.get("numeric_values", []), ensure_ascii=False),
            })
    claims_df_6b = pd.DataFrame(flat_rows_6b)
    if "numeric_values" in claims_df_6b.columns:
        claims_df_6b = claims_df_6b.drop(columns=["numeric_values"])
    claims_df_6b.to_csv(CLAIMS_OUTPUT_PATH_6B, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {CLAIMS_OUTPUT_PATH_6B} ({len(claims_df_6b):,}개 주장)")


## 6-C. Attention 기반 하이브리드 파이프라인 — 피드백 반영본

### 기존 코드 피드백

- 50건을 한 번 분할한 점수는 변동이 크고, 같은 검증셋에서 임계값을 선택해 성능이 낙관적이었습니다.
- 11:39 클래스 불균형을 학습에 반영하지 않았습니다.
- 무작위 분류 헤드를 짧게 학습한 결과보다 사전학습 attention 임베딩을 고정하고 반복 교차검증하는 편이 현재 데이터 규모에 안정적입니다.
- 마침표만 사용한 문장 분리, 앞뒤 문맥 없는 HCX 호출, 요청 실패 시 체크포인트 부재가 있었습니다.
- KOSIS 검색은 단위·기간·지역·기관을 충분히 사용하지 않았고 같은 표를 반복 임베딩했습니다.
- 정답 `tbl_id`가 없어 KOSIS 추천 정확도는 아직 평가할 수 없습니다.

### 수정 아키텍처

```text
기사 → 한국어 문장 분리 → KLUE-RoBERTa mean-pooled 임베딩
     → class_weight 적용 Logistic Regression → claim 후보
     → 후보+앞뒤 문맥을 HCX가 구조화 → KOSIS 어휘 검색
     → 캐시된 한국어 임베딩 재순위화 → 기관·통계표 Top-K
```


### 6-C-1. 독립 실행 설정과 경로

설치·학습·API 호출은 모두 기본적으로 꺼져 있습니다. 6-C 설정 셀은 `PROJECT_ROOT`나 `csv_path`가 없어도 실행됩니다.


In [ ]:
import importlib.util
import json
import random
import re
import subprocess
import sys
import time
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import requests

INSTALL_6C = False
if INSTALL_6C:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "transformers", "scikit-learn", "sentence-transformers", "joblib", "kss",
    ])

SEED_6C = 42
ENCODER_MODEL_6C = "klue/roberta-base"
RERANK_MODEL_6C = "jhgan/ko-sroberta-multitask"
MAX_LENGTH_6C = 192
BATCH_SIZE_6C = 16
CV_SPLITS_6C = 5
CV_REPEATS_6C = 3
DEFAULT_THRESHOLD_6C = 0.50
LEXICAL_TOP_K_6C = 30
FINAL_TOP_K_6C = 5
random.seed(SEED_6C)
np.random.seed(SEED_6C)

ROOT_6C = Path(globals().get("PROJECT_ROOT", Path.cwd()))
KNOWN_CSV_6C = globals().get("csv_path")
DATA_DIR_6C = Path(KNOWN_CSV_6C).parent if KNOWN_CSV_6C is not None else ROOT_6C / "data"
OUTPUT_DIR_6C = DATA_DIR_6C if DATA_DIR_6C.is_dir() else ROOT_6C
OUTPUT_STEM_6C = Path(KNOWN_CSV_6C).stem if KNOWN_CSV_6C is not None else "news_preprocessed"


def locate_file_6c(filename):
    candidates = [
        ROOT_6C / "data" / filename, ROOT_6C / filename, DATA_DIR_6C / filename,
        Path("/content/drive/MyDrive/news_verification/data") / filename,
        Path("/content/drive/MyDrive/news_verification/data/통계표") / filename,
        Path("/content") / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"{filename}을 찾지 못했습니다.\n" + "\n".join(map(str, candidates)))


LABEL_PATH_6C = locate_file_6c("candidate_labeling_pilot_relaxed_team2.csv")
TABLE_TREE_PATH_6C = locate_file_6c("kosis_table_tree.json")
ORG_NAMES_PATH_6C = locate_file_6c("kosis_org_names.json")
MODEL_DIR_6C = OUTPUT_DIR_6C / "model_6c_attention_logistic"
METRICS_PATH_6C = OUTPUT_DIR_6C / f"{OUTPUT_STEM_6C}_6C_metrics.json"
CHECKPOINT_PATH_6C = OUTPUT_DIR_6C / f"{OUTPUT_STEM_6C}_6C_checkpoint.jsonl"
OUTPUT_PATH_6C = OUTPUT_DIR_6C / f"{OUTPUT_STEM_6C}_6C_results.csv"

required_6c = ["torch", "transformers", "sklearn", "sentence_transformers", "joblib"]
missing_6c = [name for name in required_6c if importlib.util.find_spec(name) is None]
print("설치 필요:" if missing_6c else "의존성 확인 완료", missing_6c)
print("라벨:", LABEL_PATH_6C)


### 6-C-2. 수동 라벨 데이터 확인

현재 라벨은 relaxed filter를 통과한 후보 50건이므로 일반 기사 문장 전체와 분포가 다릅니다. 아래 셀은 이 제한을 수치로 기록합니다.


In [ ]:
def bool_label_6c(value):
    return int(value is True or str(value).strip().lower() in {"true", "1", "yes"})


labels_6c = pd.read_csv(LABEL_PATH_6C, encoding="utf-8-sig")
labels_6c = labels_6c.dropna(subset=["claim_text", "gold_is_aggregate_claim"]).copy()
labels_6c["label"] = labels_6c["gold_is_aggregate_claim"].map(bool_label_6c)
label_audit_6c = {
    "n": len(labels_6c), "positive": int(labels_6c["label"].sum()),
    "negative": int((labels_6c["label"] == 0).sum()),
    "unique_articles": int(labels_6c["article_idx"].nunique()) if "article_idx" in labels_6c else None,
    "all_relaxed_prefiltered": bool(labels_6c.get("passes_relaxed_filter", pd.Series(False)).fillna(False).all()),
}
print(label_audit_6c)
display(labels_6c[["sample_id", "claim_text", "label"]].head())


### 6-C-3. Attention 임베딩 + 반복 교차검증 분류기

KLUE-RoBERTa를 미세조정하지 않고 attention encoder의 mean-pooled 문장 임베딩을 사용합니다. `class_weight="balanced"` Logistic Regression을 반복 층화 교차검증하며, 고정 임계값 0.5 성능을 주지표로 저장합니다. OOF에서 고른 임계값 성능은 탐색 지표로만 표시합니다.


In [ ]:
RUN_TRAIN_6C = False


def train_attention_classifier_6c(frame):
    import joblib
    import torch
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    from sklearn.model_selection import RepeatedStratifiedKFold
    from transformers import AutoModel, AutoTokenizer

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("1/4 encoder 로드:", ENCODER_MODEL_6C, "/", device)
    tokenizer = AutoTokenizer.from_pretrained(ENCODER_MODEL_6C)
    encoder = AutoModel.from_pretrained(ENCODER_MODEL_6C).to(device).eval()

    def encode_texts(texts):
        vectors = []
        texts = list(map(str, texts))
        for start in range(0, len(texts), BATCH_SIZE_6C):
            batch = tokenizer(
                texts[start:start + BATCH_SIZE_6C], padding=True, truncation=True,
                max_length=MAX_LENGTH_6C, return_tensors="pt",
            ).to(device)
            with torch.no_grad():
                hidden = encoder(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
            vectors.append(torch.nn.functional.normalize(pooled, dim=1).cpu().numpy())
        return np.vstack(vectors)

    print("2/4 수동 라벨 문장 임베딩")
    features = encode_texts(frame["claim_text"])
    target = frame["label"].to_numpy()
    cv = RepeatedStratifiedKFold(
        n_splits=CV_SPLITS_6C, n_repeats=CV_REPEATS_6C, random_state=SEED_6C
    )
    probability_sum = np.zeros(len(frame), dtype=float)
    prediction_count = np.zeros(len(frame), dtype=int)
    print("3/4 반복 교차검증")
    for train_index, valid_index in cv.split(features, target):
        fold_model = LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=SEED_6C
        )
        fold_model.fit(features[train_index], target[train_index])
        probability_sum[valid_index] += fold_model.predict_proba(features[valid_index])[:, 1]
        prediction_count[valid_index] += 1
    oof_probability = probability_sum / np.maximum(prediction_count, 1)

    fixed_prediction = (oof_probability >= DEFAULT_THRESHOLD_6C).astype(int)
    fixed_report = classification_report(target, fixed_prediction, output_dict=True, zero_division=0)
    candidates = np.arange(0.20, 0.81, 0.05)
    exploratory_threshold = max(
        candidates,
        key=lambda threshold: f1_score(target, oof_probability >= threshold, zero_division=0),
    )
    exploratory_prediction = (oof_probability >= exploratory_threshold).astype(int)
    exploratory_report = classification_report(target, exploratory_prediction, output_dict=True, zero_division=0)
    metrics = {
        "data_audit": label_audit_6c, "cv_splits": CV_SPLITS_6C, "cv_repeats": CV_REPEATS_6C,
        "primary_threshold": DEFAULT_THRESHOLD_6C,
        "primary_precision": fixed_report["1"]["precision"],
        "primary_recall": fixed_report["1"]["recall"],
        "primary_f1": fixed_report["1"]["f1-score"],
        "primary_confusion_matrix": confusion_matrix(target, fixed_prediction).tolist(),
        "exploratory_threshold": float(exploratory_threshold),
        "exploratory_f1": exploratory_report["1"]["f1-score"],
        "warning": "탐색 임계값은 같은 OOF 예측으로 선택했으므로 최종 성능이 아님",
    }

    print("4/4 전체 라벨로 최종 분류기 학습·저장")
    classifier = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED_6C)
    classifier.fit(features, target)
    MODEL_DIR_6C.mkdir(parents=True, exist_ok=True)
    tokenizer.save_pretrained(MODEL_DIR_6C / "encoder")
    encoder.save_pretrained(MODEL_DIR_6C / "encoder")
    joblib.dump(classifier, MODEL_DIR_6C / "classifier.joblib")
    METRICS_PATH_6C.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    display(pd.DataFrame([metrics]))
    return encoder, tokenizer, classifier, device, encode_texts, metrics


if RUN_TRAIN_6C:
    encoder_6c, tokenizer_6c, classifier_6c, device_6c, encode_texts_6c, metrics_6c = train_attention_classifier_6c(labels_6c)


### 6-C-4. 모델 로드, 한국어 문장 분리, 문맥 포함 후보 탐지

`kss`가 있으면 한국어 문장 분리를 사용하고, 없으면 줄바꿈과 종결부호를 함께 사용하는 fallback을 적용합니다. HCX 전달용으로 앞뒤 한 문장을 보존합니다.


In [ ]:
RUN_LOAD_MODEL_6C = False


def load_attention_classifier_6c():
    import joblib
    import torch
    from transformers import AutoModel, AutoTokenizer
    model_path = MODEL_DIR_6C / "encoder"
    classifier_path = MODEL_DIR_6C / "classifier.joblib"
    if not model_path.is_dir() or not classifier_path.is_file():
        raise FileNotFoundError("6-C-3 학습 셀을 먼저 실행하세요.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    encoder = AutoModel.from_pretrained(model_path).to(device).eval()
    classifier = joblib.load(classifier_path)

    def encode_texts(texts):
        vectors = []
        texts = list(map(str, texts))
        for start in range(0, len(texts), BATCH_SIZE_6C):
            batch = tokenizer(
                texts[start:start + BATCH_SIZE_6C], padding=True, truncation=True,
                max_length=MAX_LENGTH_6C, return_tensors="pt",
            ).to(device)
            with torch.no_grad():
                hidden = encoder(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
            vectors.append(torch.nn.functional.normalize(pooled, dim=1).cpu().numpy())
        return np.vstack(vectors)
    return encoder, tokenizer, classifier, device, encode_texts


if RUN_LOAD_MODEL_6C:
    encoder_6c, tokenizer_6c, classifier_6c, device_6c, encode_texts_6c = load_attention_classifier_6c()


def split_sentences_6c(text):
    text = str(text).strip()
    if not text:
        return []
    if importlib.util.find_spec("kss") is not None:
        import kss
        sentences = kss.split_sentences(text)
    else:
        sentences = re.split(r"(?<=[.!?])\s+|[\r\n]+", text)
    return [re.sub(r"\s+", " ", sentence).strip() for sentence in sentences if len(sentence.strip()) >= 10]


def detect_candidates_6c(article_text, threshold=DEFAULT_THRESHOLD_6C):
    if "encode_texts_6c" not in globals() or "classifier_6c" not in globals():
        raise RuntimeError("6-C-3에서 학습하거나 6-C-4에서 저장 모델을 불러오세요.")
    sentences = split_sentences_6c(article_text)
    columns = ["sentence_no", "candidate_text", "context_text", "claim_probability"]
    if not sentences:
        return pd.DataFrame(columns=columns)
    probability = classifier_6c.predict_proba(encode_texts_6c(sentences))[:, 1]
    rows = []
    for index, (sentence, score) in enumerate(zip(sentences, probability)):
        if score < threshold:
            continue
        context = " ".join(sentences[max(0, index - 1):min(len(sentences), index + 2)])
        rows.append({
            "sentence_no": index + 1, "candidate_text": sentence,
            "context_text": context, "claim_probability": float(score),
        })
    return pd.DataFrame(rows, columns=columns)


### 6-C-5. HCX 대상 문장 구조화와 오류 복구

앞뒤 문맥은 해석에만 사용하고 `target_sentence`의 주장만 반환하도록 별도 프롬프트를 사용합니다. 기간·비교시점·지역·모집단·기관을 명시적으로 출력합니다.


In [ ]:
CLAIM_SCHEMA_6C = {
    "type": "object", "properties": {"claims": {"type": "array", "items": {
        "type": "object", "properties": {
            "claim_text": {"type": "string"}, "claim_type": {"type": "string", "enum": ["numeric", "non_numeric"]},
            "subject": {"type": "string"}, "predicate": {"type": "string"}, "object": {"type": "string"},
            "population": {"type": "string"}, "region": {"type": "string"},
            "time_ref": {"type": "array", "items": {"type": "string"}},
            "time_compare": {"type": "array", "items": {"type": "string"}},
            "source_org_raw": {"type": "string"},
            "numeric_values": {"type": "array", "items": {"type": "object", "properties": {
                "raw_value": {"type": "string"}, "normalized_value": {"type": "string"},
                "unit": {"type": "string"}, "context": {"type": "string"}
            }, "required": ["raw_value", "normalized_value", "unit", "context"]}},
            "evidence_quote": {"type": "string"}
        }, "required": ["claim_text", "claim_type", "subject", "predicate", "object", "population", "region",
                       "time_ref", "time_compare", "source_org_raw", "numeric_values", "evidence_quote"]
    }}}, "required": ["claims"]
}

SYSTEM_PROMPT_6C = """당신은 한국어 뉴스의 통계 검증 가능 주장을 구조화한다.
context는 대상 문장의 생략된 주체·시점 해석에만 사용한다.
반드시 target_sentence에 존재하는 주장만 반환하고 주변 문장의 별도 주장은 추출하지 않는다.
원문에 없는 값·단위·기관은 만들지 말고 불명확한 문자열 필드는 빈 문자열, 배열 필드는 빈 배열로 반환한다.
주관적 평가·전망·개별 기업 계획은 제외한다. JSON 스키마 외 설명을 출력하지 않는다."""


def call_hcx_6c(target_sentence, context_text, max_retries=3):
    if not NCP_API_KEY:
        raise RuntimeError("NCP_CLOVASTUDIO_API_KEY가 설정되지 않았습니다.")
    headers = {
        "Authorization": f"Bearer {NCP_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json", "Accept": "application/json",
    }
    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_6C},
            {"role": "user", "content": json.dumps({
                "target_sentence": target_sentence, "context": context_text
            }, ensure_ascii=False)},
        ],
        "topP": 0.8, "topK": 0, "temperature": 0.1, "repetitionPenalty": 1.1,
        "maxCompletionTokens": 2048, "thinking": {"effort": "none"},
        "responseFormat": {"type": "json", "schema": CLAIM_SCHEMA_6C},
    }
    for attempt in range(max_retries):
        try:
            response = requests.post(NCP_API_URL, headers=headers, json=body, timeout=180)
            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"retryable status={response.status_code}", response=response)
            response.raise_for_status()
            result = response.json().get("result", {})
            return json.loads(result["message"]["content"]), result.get("usage", {})
        except (requests.Timeout, requests.HTTPError) as error:
            if attempt + 1 == max_retries:
                raise
            time.sleep(2 ** attempt)


def structure_candidates_6c(candidate_frame):
    rows, errors = [], []
    usage_total = {"promptTokens": 0, "completionTokens": 0, "totalTokens": 0}
    for _, candidate in candidate_frame.iterrows():
        try:
            parsed, usage = call_hcx_6c(candidate["candidate_text"], candidate["context_text"])
            for claim_number, claim in enumerate(parsed.get("claims", []), 1):
                rows.append({**candidate.to_dict(), "claim_number": claim_number, **claim})
            for key in usage_total:
                usage_total[key] += int(usage.get(key, 0) or 0)
        except Exception as error:
            errors.append({**candidate.to_dict(), "error_type": type(error).__name__, "error": str(error)})
    return pd.DataFrame(rows), pd.DataFrame(errors), usage_total


### 6-C-6. KOSIS 검색과 캐시된 bi-encoder 재순위화

이 단계는 cross-encoder가 아니라 빠른 bi-encoder 의미 재순위화입니다. 지표뿐 아니라 모집단·지역·기간·기관·단위를 검색 질의에 포함하고, 한 번 계산한 통계표 임베딩은 메모리에서 재사용합니다.


In [ ]:
def load_kosis_catalog_6c():
    tree = json.loads(TABLE_TREE_PATH_6C.read_text(encoding="utf-8"))
    organizations = json.loads(ORG_NAMES_PATH_6C.read_text(encoding="utf-8"))
    rows = []
    for root in (tree.values() if isinstance(tree, dict) else tree):
        for leaf in root.get("leaves", []):
            org_id = str(leaf.get("org_id", ""))
            rows.append({
                "org_id": org_id, "org_name": str(organizations.get(org_id, "")),
                "tbl_id": str(leaf.get("tbl_id", "")), "tbl_nm": str(leaf.get("tbl_nm", "")),
                "stat_id": str(leaf.get("stat_id", "")),
                "category_path": " > ".join(map(str, leaf.get("path", []))),
            })
    catalog = pd.DataFrame(rows).drop_duplicates("tbl_id").reset_index(drop=True)
    catalog["search_document"] = (catalog["org_name"] + " " + catalog["tbl_nm"] + " " + catalog["category_path"]).str.strip()
    return catalog


TOKEN_RE_6C = re.compile(r"[가-힣A-Za-z0-9]{2,}")
STOPWORDS_6C = {"통계", "자료", "관련", "따르면", "기준", "지난해", "올해", "나타났다"}
TABLE_VECTOR_CACHE_6C = {}


def claim_query_6c(claim):
    values = claim.get("numeric_values", [])
    units = " ".join(str(item.get("unit", "")) for item in values if isinstance(item, dict))
    fields = ["claim_text", "subject", "object", "population", "region", "source_org_raw"]
    arrays = ["time_ref", "time_compare"]
    return " ".join(
        [str(claim.get(field, "")) for field in fields]
        + [" ".join(map(str, claim.get(field, []))) for field in arrays]
        + [units]
    ).strip()


def lexical_candidates_6c(query, catalog, top_k=LEXICAL_TOP_K_6C):
    tokens = {token.lower() for token in TOKEN_RE_6C.findall(query) if token.lower() not in STOPWORDS_6C}
    if not tokens:
        return catalog.iloc[:0].assign(lexical_score=pd.Series(dtype=float))
    documents = catalog["search_document"].str.lower()
    scores = pd.Series(0.0, index=catalog.index)
    for token in tokens:
        scores += documents.str.contains(re.escape(token), regex=True, na=False).astype(float)
    selected = scores[scores > 0].nlargest(top_k)
    result = catalog.loc[selected.index].copy()
    result["lexical_score"] = selected.to_numpy()
    return result


RUN_LOAD_KOSIS_6C = False
if RUN_LOAD_KOSIS_6C:
    from sentence_transformers import SentenceTransformer
    print("KOSIS catalog 로드")
    kosis_catalog_6c = load_kosis_catalog_6c()
    print("bi-encoder 로드:", RERANK_MODEL_6C)
    reranker_6c = SentenceTransformer(RERANK_MODEL_6C)


def recommend_tables_6c(claim, catalog, top_k=FINAL_TOP_K_6C):
    if "reranker_6c" not in globals():
        raise RuntimeError("RUN_LOAD_KOSIS_6C=True로 검색 모델을 로드하세요.")
    query = claim_query_6c(claim)
    candidates = lexical_candidates_6c(query, catalog)
    if candidates.empty:
        return candidates
    missing = candidates[~candidates["tbl_id"].isin(TABLE_VECTOR_CACHE_6C)]
    if not missing.empty:
        vectors = reranker_6c.encode(missing["search_document"].tolist(), normalize_embeddings=True)
        TABLE_VECTOR_CACHE_6C.update(dict(zip(missing["tbl_id"], vectors)))
    table_vectors = np.vstack([TABLE_VECTOR_CACHE_6C[tbl_id] for tbl_id in candidates["tbl_id"]])
    query_vector = reranker_6c.encode([query], normalize_embeddings=True)[0]
    candidates["semantic_score"] = table_vectors @ query_vector
    candidates["final_score"] = (
        0.3 * candidates["lexical_score"] / max(float(candidates["lexical_score"].max()), 1.0)
        + 0.7 * candidates["semantic_score"]
    )
    return candidates.sort_values("final_score", ascending=False).head(top_k).reset_index(drop=True)


### 6-C-7. 체크포인트 기반 통합 실행

기사별 성공·실패·토큰을 JSONL에 즉시 기록합니다. 동일 URL의 성공 기록은 재실행에서 제외합니다.


In [ ]:
RUN_PIPELINE_6C = False
MAX_ARTICLES_6C = 1


def completed_urls_6c(path=CHECKPOINT_PATH_6C):
    if not path.is_file():
        return set()
    completed = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            record = json.loads(line)
            if record.get("status") == "success":
                completed.add(str(record.get("url", "")))
        except json.JSONDecodeError:
            continue
    return completed


def run_pipeline_6c(news_frame, max_articles=MAX_ARTICLES_6C):
    body_column = globals().get("CLEAN_BODY_COLUMN", "본문_정제")
    finished = completed_urls_6c()
    targets = news_frame[news_frame[body_column].notna() & ~news_frame["URL"].astype(str).isin(finished)]
    if max_articles is not None:
        targets = targets.head(max_articles)
    flat_rows = []
    with CHECKPOINT_PATH_6C.open("a", encoding="utf-8") as checkpoint:
        for sequence, (article_index, article) in enumerate(targets.iterrows(), 1):
            record = {"article_index": int(article_index), "url": str(article.get("URL", "")), "title": article.get("기사제목")}
            try:
                detected = detect_candidates_6c(article[body_column])
                claims, errors, usage = structure_candidates_6c(detected)
                for _, claim in claims.iterrows():
                    claim_data = claim.to_dict()
                    recommendations = recommend_tables_6c(claim_data, kosis_catalog_6c)
                    if recommendations.empty:
                        flat_rows.append({**record, **claim_data, "kosis_rank": None})
                    for rank, (_, table) in enumerate(recommendations.iterrows(), 1):
                        flat_rows.append({
                            **record, **claim_data, "kosis_rank": rank,
                            **table[["org_id", "org_name", "tbl_id", "tbl_nm", "stat_id", "category_path",
                                     "lexical_score", "semantic_score", "final_score"]].to_dict(),
                        })
                record.update({
                    "status": "success", "candidate_count": len(detected), "claim_count": len(claims),
                    "candidate_error_count": len(errors), "usage": usage,
                })
            except Exception as error:
                record.update({"status": "failed", "error_type": type(error).__name__, "error": str(error)})
            checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
            checkpoint.flush()
            print(f"[{sequence}/{len(targets)}] {record['status']} - {record['title']}")
    results = pd.DataFrame(flat_rows)
    if not results.empty:
        # 체크포인트 재개 시 이전 CSV를 덮어쓰지 않고 새 결과만 이어 씁니다.
        results.to_csv(
            OUTPUT_PATH_6C, mode="a", header=not OUTPUT_PATH_6C.is_file(),
            index=False, encoding="utf-8-sig",
        )
    return results


if RUN_PIPELINE_6C:
    if not NCP_API_KEY:
        raise RuntimeError("NCP_CLOVASTUDIO_API_KEY가 필요합니다.")
    results_6c = run_pipeline_6c(df)
    display(results_6c.head(20))


### 6-C-8. 완료 기준과 남은 제한

- 주장 탐지는 반복 교차검증의 고정 임계값 Precision·Recall·F1을 주지표로 사용합니다.
- 탐색 임계값 점수는 참고용이며 독립 테스트 성능으로 보고하지 않습니다.
- HCX는 필드별 정답 라벨을 추가한 뒤 값·단위·기간·기관 정확도와 JSON 실패율을 평가합니다.
- KOSIS 추천은 정답 기관·`tbl_id`를 라벨링하기 전에는 성능 평가가 아니라 후보 산출 단계입니다.
- 현재 50건이 모두 사전 필터 후보이므로, 실제 배포 전 일반 비주장 문장을 추가 라벨링해야 합니다.


## 7. 기사 주장 추출 실험

### 7-1. Claim 후보 주장 탐지

In [59]:
"""
# 단계별 Claim 추출 파이프라인

1단계: 규칙 기반 후보 탐지 (높은 Recall)
2단계: LLM 기반 Claim 구조화
3단계: 규칙 기반 후검증
4단계: 통계표 매칭 가능성 판정
"""

import re
from typing import NamedTuple
from dataclasses import dataclass, field, asdict

# ============ 1단계: 규칙 기반 후보 탐지 ============

class NumericCandidate(NamedTuple):
    sentence: str
    start_pos: int
    end_pos: int
    reason: str  # 'numeric', 'unit', 'comparison', 'rank', 'attribution'
    matched_text: str

# 숫자 패턴
NUMERIC_RE = re.compile(
    r'\d{1,3}(?:[,.]?\d{3})*(?:[.]\d+)?'  # 천 단위 쉼표/점, 소수점
    r'|약\s*\d+|대략\s*\d+|거의\s*\d+'  # 근사 표현
    r'|[일이삼사오육칠팔구십백천만억조]+'  # 한글 숫자
)

# 단위 패턴
UNIT_RE = re.compile(
    r'(?:명|건|건|석|톤|원|달러|유로|엔|위안|'  # 기본 단위
    r'%|퍼센트|배|배수|배율|'  # 비율/배수
    r'년|월|일|시간|분|초|주|분기|'  # 시간 단위
    r'자리|등급|순|'  # 서수/순위
    r'대/직원|명/직원|건/직원|'  # 복합 단위
    r'개/인|마리|제곱미터|입방미터|'
    r'km|m|cm|kg|g|ml|l|'  # 길이/무게/용량
    r'km/h|m/s|'  # 속도
    r'도|℃|℉|'  # 온도
    r'dB|db|'  # 음량
    r'점|스코어|상승|하락|증가|감소'  # 변화 표현
    r')(?:\s|$|[.,!?])'
)

# 비교 패턴
COMPARISON_RE = re.compile(
    r'(?:보다|에\s*비[해혹]|초과|미만|이상|이하|'
    r'더|다|적|많|높|낮|크|작|'
    r'~배|%\s*(?:증가|감소|상승|하락|'
    r'를?\s*(?:넘|초과|이상))'
    r')'
)

# 순위/최상위/최하위 패턴
RANK_RE = re.compile(
    r'(?:1위|2위|3위|순위|최고|최저|최상|최하|1번|2번|3번|'
    r'첫번째|두번째|세번째|'
    r'가장|제일|'
    r'우선|이후|다음으로)'
)

# 출처/속성 패턴
ATTRIBUTION_RE = re.compile(
    r"(?:통계청|KOSIS|조사|발표|자료|통계|"
    r"에 따르면|는|가|에서 말한|에서는|이 나타냈|이 조사|"
    r"보도|언급|지적|지표|수치)"
)

def extract_numeric_candidates(text: str, min_context: int = 30) -> list[NumericCandidate]:
    """높은 Recall로 숫자 후보를 탐지합니다."""
    candidates = []
    sentences = re.split(r'[.!?]+', text)
    
    char_offset = 0
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            char_offset += len(sentence) + 1
            continue
        
        # 숫자 매칭
        for match in NUMERIC_RE.finditer(sentence):
            start = max(0, match.start() - min_context)
            end = min(len(sentence), match.end() + min_context)
            candidates.append(NumericCandidate(
                sentence=sentence,
                start_pos=char_offset + start,
                end_pos=char_offset + end,
                reason='numeric',
                matched_text=match.group()
            ))
        
        # 단위 매칭
        for match in UNIT_RE.finditer(sentence):
            start = max(0, match.start() - min_context)
            end = min(len(sentence), match.end() + min_context)
            candidates.append(NumericCandidate(
                sentence=sentence,
                start_pos=char_offset + start,
                end_pos=char_offset + end,
                reason='unit',
                matched_text=match.group()
            ))
        
        # 비교 매칭
        if COMPARISON_RE.search(sentence):
            candidates.append(NumericCandidate(
                sentence=sentence,
                start_pos=char_offset,
                end_pos=char_offset + len(sentence),
                reason='comparison',
                matched_text=sentence[:50]
            ))
        
        # 순위 매칭
        if RANK_RE.search(sentence):
            candidates.append(NumericCandidate(
                sentence=sentence,
                start_pos=char_offset,
                end_pos=char_offset + len(sentence),
                reason='rank',
                matched_text=sentence[:50]
            ))
        
        # 출처/속성 매칭
        if ATTRIBUTION_RE.search(sentence):
            candidates.append(NumericCandidate(
                sentence=sentence,
                start_pos=char_offset,
                end_pos=char_offset + len(sentence),
                reason='attribution',
                matched_text=sentence[:50]
            ))
        
        char_offset += len(sentence) + 1
    
    return candidates

# 테스트
test_article = df[df[CLEAN_BODY_COLUMN].notna()].iloc[0][CLEAN_BODY_COLUMN]
test_candidates = extract_numeric_candidates(test_article[:2000])
print(f"탐지된 후보: {len(test_candidates)}개")
display(pd.DataFrame([{
    'reason': c.reason,
    'matched_text': c.matched_text,
    'sentence': c.sentence[:80]
} for c in test_candidates[:20]]))

탐지된 후보: 100개


,reason,matched_text,sentence
0,numeric,사,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다
1,numeric,일,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다
2,numeric,9,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다
3,numeric,이,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다
4,numeric,구조,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다
5,comparison,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다
6,attribution,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다
7,numeric,31,동물권보호단체 케어는 지난달 31일 공식 인스타그램을 통해 보호자 없이 마을을 배회하던 푸딩이를 안전하게 보호 중이라고 밝혔다
8,numeric,일,동물권보호단체 케어는 지난달 31일 공식 인스타그램을 통해 보호자 없이 마을을 배회하던 푸딩이를 안전하게 보호 중이라고 밝혔다
9,numeric,이,동물권보호단체 케어는 지난달 31일 공식 인스타그램을 통해 보호자 없이 마을을 배회하던 푸딩이를 안전하게 보호 중이라고 밝혔다


### 7-2. 다단계 Claim 추출 파이프라인

2단계: LLM 기반 Claim 구조화 및 3·4단계 후검증


In [73]:
def structure_claims_with_llm(
    article_text: str,
    candidates: list[NumericCandidate],
    include_summary: bool = True
) -> tuple[list[ExtractedClaim], str]:
    """
    후보 문장들에서 수치가 포함된 맥락의 핵심 문장을 추출하고 구조화한 후, 요약을 생성합니다.
    
    Returns:
        tuple: (추출된 claims 리스트, 요약 텍스트)
    """
    if not NCP_API_KEY or not candidates:
        return [], ""
    
    # 후보 문장만 묶어서 전달
    candidate_sentences = list(set(c.sentence for c in candidates))
    candidate_text = "\n".join(candidate_sentences[:50])  # 최대 50개 문장
    
    headers = {
        "Authorization": f"Bearer {NCP_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    
    # ===== 단계 1: 핵심 문장 추출 =====
    extract_prompt = """다음 문장들에서 수치(숫자, 통계, 수량 등)를 포함하고 있는 핵심 문장들을 추출하세요.

추출 규칙:
1. 수치를 명확하게 포함하는 문장만 선택
2. 맥락을 충분히 유지하면서도 간결한 핵심 문장
3. 문장의 주제, 지표, 수치, 단위, 시간이 가능한 한 명확해야 함
4. 불필요한 수식이나 부연설명은 제거

출력 형식 (텍스트, 한 줄에 하나):
- 각 문장을 그대로 한 줄씩 나열"""
    
    extract_body = {
        "messages": [
            {"role": "system", "content": extract_prompt},
            {"role": "user", "content": f"다음 후보 문장들에서 수치가 포함된 핵심 문장을 추출하세요:\n\n{candidate_text}"}
        ],
        "topP": 0.7,
        "topK": 0,
        "temperature": 0.1,
        "maxCompletionTokens": 2048,
        "thinking": {"effort": "none"},
    }
    
    extracted_sentences = ""
    claims = []
    
    try:
        # 단계 1: 핵심 문장 추출
        response1 = requests.post(NCP_API_URL, headers=headers, json=extract_body, timeout=120)
        response1.raise_for_status()
        
        payload1 = response1.json()
        result1 = payload1.get("result", {})
        message1 = result1.get("message", {})
        extracted_sentences = message1.get("content", "")
        
        if not extracted_sentences and "thinkingContent" in message1:
            extracted_sentences = message1.get("thinkingContent", "")
        
        if not extracted_sentences:
            print("⚠️ 경고: API 응답에서 content를 추출할 수 없습니다.")
            return [], ""
        
        # 추출된 문장들을 ExtractedClaim으로 변환
        for line in extracted_sentences.strip().split("\n"):
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            
            # 원문에서 매칭되는 후보 찾기
            matching_candidate = None
            for cand in candidates:
                if cand.sentence in line or line in cand.sentence:
                    matching_candidate = cand
                    break
            
            # ExtractedClaim 생성
            claims.append(ExtractedClaim(
                claim_text=line,
                claim_type=matching_candidate.reason if matching_candidate else "numeric",
                subject="",
                metric="",
                temporal="",
                value=None,
                unit=None,
                confidence=0.7,
            ))
        
        # ===== 단계 2: 요약 생성 =====
        summary_text = ""
        if include_summary and extracted_sentences:
            headers["X-NCP-CLOVASTUDIO-REQUEST-ID"] = str(uuid.uuid4())
            
            summary_prompt = """다음 핵심 문장들을 기반으로 간결하고 명확한 요약을 작성하세요.
                요약 규칙:
                - 수치와 통계 정보를 명확하게 포함
                - 3~5개 문장의 간결한 요약
                - 핵심 내용만 전달
                - 한국어로 자연스럽게 작성"""
            
            summary_body = {
                "messages": [
                    {"role": "system", "content": summary_prompt},
                    {"role": "user", "content": f"다음 핵심 문장들을 요약해주세요:\n\n{extracted_sentences}"}
                ],
                "topP": 0.7,
                "topK": 0,
                "temperature": 0.3,
                "maxCompletionTokens": 512,
                "thinking": {"effort": "none"},
            }
            
            try:
                response2 = requests.post(NCP_API_URL, headers=headers, json=summary_body, timeout=120)
                response2.raise_for_status()
                
                payload2 = response2.json()
                result2 = payload2.get("result", {})
                message2 = result2.get("message", {})
                summary_text = message2.get("content", "")
                
                if not summary_text and "thinkingContent" in message2:
                    summary_text = message2.get("thinkingContent", "")
            
            except requests.exceptions.RequestException as e:
                print(f"⚠️ 요약 생성 실패: {e}")
                summary_text = ""
        
        return claims, summary_text
    
    except requests.exceptions.RequestException as e:
        print(f"❌ NCP API 호출 실패: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"응답: {e.response.text}")
        return [], ""

### 7-2a. 간단한 API 테스트


In [81]:
# 핵심 문장 추출 테스트 (단계 1만)
if NCP_API_KEY:
    print("=" * 60)
    print("NCP CLOVA API 테스트 - [단계 1] 핵심 문장 추출")
    print("=" * 60)
    
    test_text = """2026년 4월 20일(월) 신문구독 | English | 日本語 | 中文 1 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선일보 산&트래블 이코노미조선 신문은 선생님 THE BOUTIQUE 행복플러스 주간조선 여성조선 월간조선 민학수 올댓골프 Topclass Life&Learning English 日本語 中文 검색 2026년 4월 20일(월) 신문구독 | English | 日本語 | 中文 1 신문구독 | English | 日本語 | 中文 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선일보 산&트래블 이코노미조선 신문은 선생님 THE BOUTIQUE 행복플러스 주간조선 여성조선 월간조선 민학수 올댓골프 Topclass Life&Learning 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선일보 산&트래블 이코노미조선 신문은 선생님 THE BOUTIQUE 행복플러스 주간조선 여성조선 월간조선 민학수 올댓골프 Topclass Life&Learning 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선일보 산&트래블 이코노미조선 신문은 선생님 THE BOUTIQUE 행복플러스 주간조선 여성조선 월간조선 민학수 올댓골프 Topclass Life&Learning 조선경제 산업·재계 대기업 63%, 올해 '1300원대' 환율 예상..."사업전략 수정 불가피" 1 국내 주요 대기업 10곳 중 6곳은 2025년 원·달러 환율 범위를 1300원대로 예상하고 올해 사업계획을 짠 것으로 나타났다. 최근 국내 정치 불안이 이어지고, 오는 20일 미국 대통령 취임을 앞둔 트럼프 당선인의 무역정책 리스크 영향으로 환율이 급등해 사업계획 수정이 불가피하다는 분석이다. 원·달러 환율은 9일 오전 8시56분 기준 1460원대를 기록하고 있다. 대한상공회의소가 최근 국내 50대 기업을 대상으로 한 '주요 대기업의 환율 영향 조사'에 따르면, 기업들이 올해 사업계획을 수립하며 적용한 원·달러 환율 범위는 1350~1400원이 33.3%로 가장 많았다. 1300~1350원 범위(29.6%)가 그 뒤를 이었다. 주요 대기업 10곳 중 6곳은 올해 사업계획을 짤 때 1300원대 환율을 적용한 셈이다. 그러나 원·달러 환율은 작년 12월 초 비상계엄 사태 이후 1430원대까지 오른 뒤, 작년 말에는 1500원대에 육박했다. 대한상의 조사 대상 기업 중 1400~1450원 범위의 환율을 적용한 기업은 18.5%였고, 현재 수준인 1450~1500원 범위로 환율을 예측하고 적용한 기업은 10곳 중 1곳(11.1%)에 불과했다. 기업들은 환율 리스크 관련해 원자재 조달 비용 증가를 가장 크게 우려했다. 기업들이 겪는 어려움을 5점 척도로 조사한 결과 '원자재·부품 조달 비용 증가'(3.70점)가 1위였고, '해외투자 비용 증가'(3.30점), '수입 결제 때 환차손 발생'(3.15점), '외화차입금 상환 부담 증가'(2.93점) 순이었다. 기업들은 환율 불안을 더 키울 수 있는 잠재 요인으로는 '국내 정치 불안정 지속'(85.2%)과 '트럼프 정부의 무역정책 본격화'(74.1%)를 꼽았다. '미국 금리 인하 지연·축소', '국내 외환 관리 불균형', '한국 국가신용평가 하락, '미국 경제 강세 지속으로 인한 달러화 가치 상승 확대' 등도 리스크가 될 것으로 전망했다. #코스닥 #트럼프 환율 대한상의 코스닥"""
    
    headers = {
        "Authorization": f"Bearer {NCP_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    
    print("\n🔹 수치 포함 핵심 문장 추출")
    print("-" * 60)
    
    extract_prompt = """다음 텍스트에서 수치(숫자, 통계, 수량 등)를 포함하고 있는 핵심 문장들을 추출하세요.
각 문장을 한 줄씩 추출하되, 다음 형식을 따르세요:
- 문장의 맥락을 유지하며 완전한 문장으로 제시
- 수치를 포함한 부분이 명확하게 드러나야 함
- 불필요한 설명 없이 문장만 나열
- 한 줄에 하나의 문장"""
    
    extract_body = {
        "messages": [
            {"role": "system", "content": extract_prompt},
            {"role": "user", "content": f"다음 텍스트에서 수치가 포함된 핵심 문장들을 추출해주세요:\n\n{test_text}"}
        ],
        "topP": 0.7,
        "topK": 0,
        "temperature": 0.1,
        "maxCompletionTokens": 1024,
        "thinking": {"effort": "none"},
    }
    
    try:
        response1 = requests.post(NCP_API_URL, headers=headers, json=extract_body, timeout=30)
        print(f"📊 응답 상태: {response1.status_code}")
        
        if response1.status_code == 200:
            payload1 = response1.json()
            extracted_sentences = payload1.get("result", {}).get("message", {}).get("content", "")
            
            if extracted_sentences:
                print("✅ 핵심 문장 추출 성공!")
                print(f"  - 추출된 문장 길이: {len(extracted_sentences)} 글자\n")
                print("📝 추출된 문장:")
                print("-" * 60)
                print(extracted_sentences)
                print("-" * 60)
            else:
                print("⚠️ 추출된 문장이 없습니다.")
                extracted_sentences = ""
        else:
            print(f"❌ 추출 실패 (상태: {response1.status_code})")
    except Exception as e:
        print(f"❌ 추출 중 오류: {e}")
        extracted_sentences = ""

else:
    print("❌ NCP API 키가 설정되지 않았습니다.")

SyntaxError: invalid syntax (2020370473.py, line 7)

In [78]:
# 핵심 문장 기반 요약 테스트 (단계 2)
if NCP_API_KEY and 'extracted_sentences' in locals() and extracted_sentences:
    print("=" * 60)
    print("NCP CLOVA API 테스트 - [단계 2] 핵심 문장 기반 요약")
    print("=" * 60)
    
    headers = {
        "Authorization": f"Bearer {NCP_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    
    print("\n🔹 핵심 문장 기반 요약 생성")
    print("-" * 60)
    
    summary_prompt = """다음 핵심 문장들을 기반으로 간결하고 명확한 요약을 작성하세요.
요약 규칙:
- 수치와 통계 정보를 명확하게 포함
- 3~5개 문장의 간결한 요약
- 핵심 내용만 전달
- 한국어로 자연스럽게 작성"""
    
    summary_body = {
        "messages": [
            {"role": "system", "content": summary_prompt},
            {"role": "user", "content": f"다음 핵심 문장들을 요약해주세요:\n\n{extracted_sentences}"}
        ],
        "topP": 0.7,
        "topK": 0,
        "temperature": 0.3,
        "maxCompletionTokens": 512,
        "thinking": {"effort": "none"},
    }
    
    try:
        response2 = requests.post(NCP_API_URL, headers=headers, json=summary_body, timeout=30)
        print(f"📊 요약 응답 상태: {response2.status_code}")
        
        if response2.status_code == 200:
            payload2 = response2.json()
            message2 = payload2.get("result", {}).get("message", {})
            summary_content = message2.get("content", "")
            thinking_content = message2.get("thinkingContent", "")
            
            print(f"  - content 길이: {len(summary_content)} 글자")
            print(f"  - thinkingContent 길이: {len(thinking_content)} 글자")
            
            if summary_content:
                print("\n✅ 요약 생성 성공!")
                print("\n📋 최종 요약:")
                print("-" * 60)
                print(summary_content)
                print("-" * 60)
                print("\n✅ 추출 및 요약 완료!")
            elif thinking_content:
                print("\n⚠️ content가 비었지만 thinking 내용이 있습니다:")
                print(thinking_content[:500])
            else:
                print("\n⚠️ 요약 내용이 없습니다.")
        else:
            print(f"❌ 요약 실패 (상태: {response2.status_code})")
            print(f"  - 응답: {response2.text[:500]}")
    except Exception as e:
        print(f"❌ 요약 중 오류: {type(e).__name__}: {e}")
    
    print("=" * 60)
else:
    print("⚠️ 먼저 7-2a 셀에서 단계 1을 실행해주세요.")

NCP CLOVA API 테스트 - [단계 2] 핵심 문장 기반 요약

🔹 핵심 문장 기반 요약 생성
------------------------------------------------------------
📊 요약 응답 상태: 200
  - content 길이: 280 글자
  - thinkingContent 길이: 0 글자

✅ 요약 생성 성공!

📋 최종 요약:
------------------------------------------------------------
현대차그룹은 작년 미국 시장에서 총 170만 8293대를 판매하여 전년 대비 3.4% 증가했으며, 현대차는 제네시스를 포함해 91만 1805대로 사상 처음 90만 대를 넘겼다. 기아는 같은 기간 동안 79만 6488대를 판매해 1.8% 증가했다. 특히 현대차의 전기차 아이오닉 5는 31%, 대형 SUV 팰리세이드는 23%의 판매 증가율을 보였고, 기아의 인기 차종으로는 EV9, 스포티지 등이 있었다. 이로써 현대차그룹은 미국 시장에서 GM, 도요타, 포드에 이어 네 번째로 많은 판매량을 기록했다.
------------------------------------------------------------

✅ 추출 및 요약 완료!


### 7-3. 배치 추출 및 결과 저장


In [ ]:
MULTISTAGE_OUTPUT_PATH = csv_path.with_name(f"{csv_path.stem}_multistage추출.jsonl")
MULTISTAGE_SUMMARY_PATH = csv_path.with_name(f"{csv_path.stem}_multistage결과.csv")

def load_completed_urls_multistage(path: Path) -> set[str]:
    if not path.is_file():
        return set()
    completed = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            record = json.loads(line)
            if record.get("status") == "success":
                completed.add(record.get("url", ""))
        except json.JSONDecodeError:
            continue
    return completed

if not NCP_API_KEY:
    print("API 키를 입력한 뒤 이 셀을 실행하세요.")
else:
    MAX_ARTICLES_MULTISTAGE = 5  # 먼저 소량으로 테스트
    
    completed_urls = load_completed_urls_multistage(MULTISTAGE_OUTPUT_PATH)
    targets = df[df[CLEAN_BODY_COLUMN].notna() & ~df["URL"].isin(completed_urls)]
    if MAX_ARTICLES_MULTISTAGE is not None:
        targets = targets.head(MAX_ARTICLES_MULTISTAGE)
    
    print(f"이번 실행 대상: {len(targets):,}건 / 기존 완료: {len(completed_urls):,}건")
    
    with MULTISTAGE_OUTPUT_PATH.open("a", encoding="utf-8") as checkpoint:
        for sequence, (_, row) in enumerate(targets.iterrows(), 1):
            record = {"url": row["URL"], "title": row["기사제목"]}
            try:
                claims, meta = extract_claims_multistage(row[CLEAN_BODY_COLUMN])
                record.update({
                    "status": "success",
                    "metadata": meta,
                    "claims": [asdict(c) for c in claims]
                })
                print(f"[{sequence}/{len(targets)}] 성공: {meta['verifiable_claims']}개 검증가능 - {row['기사제목'][:40]}")
            except Exception as error:
                record.update({
                    "status": "failed",
                    "error": str(error),
                    "metadata": {},
                    "claims": []
                })
                print(f"[{sequence}/{len(targets)}] 실패: {error}")
            
            checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
            checkpoint.flush()
            time.sleep(REQUEST_INTERVAL_SECONDS)
    
    # 결과를 CSV로 변환
    flat_rows = []
    for line in MULTISTAGE_OUTPUT_PATH.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        if record.get("status") != "success":
            continue
        
        for claim_idx, claim in enumerate(record.get("claims", []), 1):
            flat_rows.append({
                "URL": record["url"],
                "기사제목": record["title"],
                "claim_number": claim_idx,
                **claim,
            })
    
    results_df = pd.DataFrame(flat_rows)
    results_df.to_csv(MULTISTAGE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
    print(f"\n저장 완료:")
    print(f"  - 상세 JSONL: {MULTISTAGE_OUTPUT_PATH}")
    print(f"  - 요약 CSV: {MULTISTAGE_SUMMARY_PATH}")
    print(f"  - 총 {len(results_df):,}개 주장 추출")


### 7-4. 다단계 추출 결과 분석


In [ ]:
# 기존 결과 파일이 있으면 로드하여 분석
if MULTISTAGE_SUMMARY_PATH.is_file():
    summary_df = pd.read_csv(MULTISTAGE_SUMMARY_PATH, encoding="utf-8-sig")
    
    print("===== 다단계 추출 결과 요약 =====\n")
    
    # 1. 기본 통계
    print(f"총 기사: {summary_df['기사제목'].nunique()}개")
    print(f"총 주장: {len(summary_df)}개")
    print(f"기사당 평균 주장: {len(summary_df) / summary_df['기사제목'].nunique():.1f}개")
    print()
    
    # 2. Claim 유형별 분포
    print("### Claim 유형별 분포")
    claim_type_dist = summary_df['claim_type'].value_counts()
    print(claim_type_dist.to_string())
    print()
    
    # 3. 검증 가능성
    print("### 검증 가능성")
    verifiable_count = summary_df['verifiable_with_statistics'].sum()
    verifiable_pct = (verifiable_count / len(summary_df)) * 100
    print(f"검증 가능한 Claim: {verifiable_count}개 ({verifiable_pct:.1f}%)")
    print(f"판단불가 예상률: {100 - verifiable_pct:.1f}%")
    print()
    
    # 4. 필드 완성도
    print("### 필드 완성도")
    field_completion = pd.DataFrame({
        'subject 있음': [summary_df['subject'].notna().sum()],
        'metric 있음': [summary_df['metric'].notna().sum()],
        'temporal 있음': [summary_df['temporal'].notna().sum()],
        'value 있음': [summary_df['value'].notna().sum()],
        'unit 있음': [summary_df['unit'].notna().sum()],
        '필수필드 완성': [summary_df['has_all_required_fields'].sum()],
    })
    print((field_completion / len(summary_df) * 100).round(1).to_string())
    print()
    
    # 5. 검증 점수 분포
    print("### 검증 점수 분포")
    print(summary_df['verification_score'].describe().round(3).to_string())
    print()
    
    # 6. 상위 주장 (높은 검증 점수)
    print("### 높은 검증 점수 주장 (Top 10)")
    top_claims = summary_df.nlargest(10, 'verification_score')[
        ['기사제목', 'claim_text', 'subject', 'metric', 'value', 'unit', 'verification_score']
    ]
    display(top_claims)
    print()
    
    # 7. 낮은 검증 점수 주장 (분석 필요)
    print("### 낮은 검증 점수 주장 (Bottom 10) - 개선 포인트")
    bottom_claims = summary_df.nsmallest(10, 'verification_score')[
        ['claim_text', 'subject', 'metric', 'value', 'unit', 'has_all_required_fields', 'verification_score']
    ]
    display(bottom_claims)
else:
    print("아직 결과 파일이 없습니다. 위의 배치 추출 셀을 먼저 실행하세요.")


### 7-5. KOSIS 통계표 매칭 가능성 시뮬레이션


In [ ]:
# KOSIS 매칭 시뮬레이션
# 실제로는 KOSIS 메타데이터와 비교하지만, 여기서는 규칙 기반으로 시뮬레이션

if MULTISTAGE_SUMMARY_PATH.is_file():
    summary_df = pd.read_csv(MULTISTAGE_SUMMARY_PATH, encoding="utf-8-sig")
    
    def simulate_kosis_matching(row: pd.Series) -> dict:
        """
        KOSIS 통계표 매칭 가능성을 판정합니다.
        실제 구현에서는 KOSIS 메타데이터와 유사도 계산을 수행합니다.
        """
        match_scores = {}
        
        # 1. 지표명 표준화 (예: "산업용 로봇" -> "산업용로봇밀도")
        metric_normalized = row['metric'].replace(" ", "") if pd.notna(row['metric']) else ""
        
        # 2. 대상 표준화 (예: "한국" -> "대한민국", "서울시" -> "서울특별시")
        subject_normalized = row['subject'] if pd.notna(row['subject']) else ""
        
        # 3. 시점 파싱 (예: "2023년" -> year=2023, "분기" -> quarterly)
        temporal_str = row['temporal'] if pd.notna(row['temporal']) else ""
        has_year = bool(re.search(r'\d{4}년?', temporal_str))
        has_month = bool(re.search(r'\d{1,2}월', temporal_str))
        has_quarter = bool(re.search(r'분기', temporal_str))
        
        # 4. 값과 단위 완성도
        has_value = pd.notna(row['value']) and row['value'].strip() != ""
        has_unit = pd.notna(row['unit']) and row['unit'].strip() != ""
        
        # 5. 매칭 가능성 점수 계산
        matching_score = 0.0
        matching_details = []
        
        if metric_normalized:
            matching_score += 0.2
            matching_details.append("metric_ok")
        if subject_normalized:
            matching_score += 0.2
            matching_details.append("subject_ok")
        if has_year or has_month or has_quarter:
            matching_score += 0.2
            matching_details.append("temporal_ok")
        if has_value:
            matching_score += 0.2
            matching_details.append("value_ok")
        if has_unit:
            matching_score += 0.2
            matching_details.append("unit_ok")
        
        return {
            "matching_score": matching_score,
            "can_match": matching_score >= 0.8,  # 80% 이상이면 매칭 가능한 것으로 판정
            "matching_details": ",".join(matching_details),
        }
    
    # 모든 Claim에 KOSIS 매칭 점수 계산
    kosis_results = summary_df.apply(simulate_kosis_matching, axis=1, result_type='expand')
    summary_df = pd.concat([summary_df, kosis_results], axis=1)
    
    # 결과 저장
    KOSIS_MATCH_PATH = csv_path.with_name(f"{csv_path.stem}_kosis매칭.csv")
    summary_df.to_csv(KOSIS_MATCH_PATH, index=False, encoding="utf-8-sig")
    
    print("===== KOSIS 매칭 가능성 분석 =====\n")
    
    # 매칭 가능한 주장 분포
    can_match_count = summary_df['can_match'].sum()
    can_match_pct = (can_match_count / len(summary_df)) * 100
    print(f"KOSIS 매칭 가능한 Claim: {can_match_count}개 ({can_match_pct:.1f}%)")
    print(f"매칭 필요한 Claim: {len(summary_df) - can_match_count}개 ({100 - can_match_pct:.1f}%)")
    print()
    
    # 매칭 점수 분포
    print("### KOSIS 매칭 점수 분포")
    print(summary_df['matching_score'].describe().round(3).to_string())
    print()
    
    # 매칭 세부 사항
    print("### 매칭 요소 완성도")
    detail_counts = {}
    for details in summary_df['matching_details']:
        for detail in str(details).split(","):
            detail_counts[detail] = detail_counts.get(detail, 0) + 1
    for detail, count in sorted(detail_counts.items()):
        print(f"  {detail}: {count}개 ({count/len(summary_df)*100:.1f}%)")
    print()
    
    # 매칭 가능한 상위 주장
    print("### 높은 매칭 점수 주장 (KOSIS 검증 우선순위)")
    priority_claims = summary_df[summary_df['can_match']].nlargest(10, 'matching_score')[
        ['기사제목', 'claim_text', 'subject', 'metric', 'temporal', 'value', 'unit', 'matching_score']
    ]
    display(priority_claims)
    print()
    
    # 매칭 불가능한 주장 (개선 필요)
    print("### 낮은 매칭 점수 주장 (개선 필요)")
    improvement_claims = summary_df[~summary_df['can_match']].nsmallest(10, 'matching_score')[
        ['claim_text', 'subject', 'metric', 'temporal', 'value', 'unit', 'matching_score']
    ]
    display(improvement_claims)
    
    print(f"\n저장 완료: {KOSIS_MATCH_PATH}")
else:
    print("아직 결과 파일이 없습니다. 위의 배치 추출 셀을 먼저 실행하세요.")


### 7-6. 다단계 파이프라인 개선 가이드

다음 항목들을 개선하면 추출 정확도와 KOSIS 매칭율을 높일 수 있습니다.


In [ ]:
"""
다단계 파이프라인 개선 전략
"""

improvement_guide = """
# 1단계: 규칙 기반 후보 탐지 개선

## 현재 문제
- 범위가 넓어 False Positive 포함 (의도된 설계)
- 시점 표현 다양성 미처리 (상대 시점, 예상 시점 등)

## 개선 방안
- 시점 패턴 확대: "올해", "내년", "작년", "분기별", "월별" 등
- 기간 표현: "2020~2023년", "2023년 1월~6월" 등
- 맥락 기반 필터: 뉴스 발행일을 기준으로 상대 시점 정규화

---

# 2단계: LLM 기반 구조화 개선

## 현재 문제
- 복합 문장에서 주어 생략 처리 미흡
- 지표명 다양성 (동일 지표의 여러 표현: "밀도", "대당 수", "인당 수")
- 단위 정규화 부족

## 개선 방안
- Prompt 엔지니어링: 문장 분리 규칙 명확화
- 지표 사전 구축: KOSIS 메타데이터 기반 지표 매핑
- 단위 정규화: "명/명", "대/직원" 등 복합 단위 표준화
- 신뢰도 보정: "확신 못함 (confidence 0.3)" vs "명확함 (confidence 0.9)" 구분

---

# 3단계: 규칙 기반 후검증 개선

## 현재 문제
- 필드별 검증만 수행 (필드 간 논리적 일관성 미확인)
- 숫자 형식 다양성 미처리 ("약 1000명", "1,234.5개" 등)

## 개선 방안
- 숫자 정규화: "약 1,234" → "1234" (정확도 90% 이상)
- 단위 일관성 검증: 지표에 맞는 단위 확인
- 시점 유효성: 기사 발행일 이후 시점 제거
- 값 범위 검증: 물리적으로 불가능한 값 필터링

---

# 4단계: KOSIS 매칭 개선

## 현재 문제
- 단순 점수 계산 (실제로는 시맨틱 매칭 필요)
- KOSIS 메타데이터 미활용

## 개선 방안
1. **지표명 매칭** (0.3점)
   - KOSIS 표 이름과 Claim 지표 유사도
   - 각 표의 실제 항목(item)과 비교
   
2. **대상 매칭** (0.2점)
   - 지역: "서울" → "서울특별시" 정규화
   - 집단: "남성" → "성별=남" 매핑
   
3. **시점 매칭** (0.2점)
   - 표의 수록 시점과 비교
   - 상대 시점을 기사 발행일 기준으로 변환
   
4. **단위 매칭** (0.15점)
   - 표의 수치 단위와 Claim 단위 일치 확인
   
5. **신뢰도 보정** (0.15점)
   - LLM 신뢰도와 검증 점수 결합

## 우선순위
1. 지표명 매칭 강화 (가장 높은 ROI)
2. KOSIS 메타데이터 연동 (KOSIS API 활용)
3. 시점 정규화 (기사 발행일 기반)

---

# 일반적인 개선 순서

1. **기준선 수립**: 현재 파이프라인 성과 측정 (precision, recall, F1)
2. **병목 분석**: 어느 단계에서 가장 많은 손실 발생?
3. **우선순위 개선**: ROI 높은 순서대로
4. **테스트 셋 검증**: 실제 뉴스 기사로 효과 검증
5. **반복 개선**: 상용 배포 전 여러 주기 개선

---

# 추가 고려사항

## 비용-효율 트레이드오프
- LLM API 호출 비용 vs 정확도
- → 규칙 기반 필터로 확실한 True Positive만 LLM으로 처리

## 운영 모니터링
- 매월 추출된 주장의 검증 가능률 추적
- KOSIS와의 실제 매칭 성공률 모니터링
- 실패 사례 분석 및 피드백 루프 구축

## 도메인 특화
- 뉴스 도메인별 지표 사전 구축
  - 경제: 산업별 통계
  - 사회: 인구통계, 범죄
  - 환경: 오염도, 온도 등
"""

print(improvement_guide)

# 권장 우선순위
print("\n" + "="*60)
print("### 추천 개선 우선순위 (ROI 기준)")
print("="*60)

priority_table = pd.DataFrame([
    {
        "순위": 1,
        "개선항목": "지표명 표준화 사전 구축",
        "난이도": "중",
        "예상효과": "매칭율 +20-30%",
        "비용": "중",
        "시간": "1-2주"
    },
    {
        "순위": 2,
        "개선항목": "시점 표현 정규화",
        "난이도": "중",
        "예상효과": "검증율 +15-20%",
        "비용": "저",
        "시간": "3-5일"
    },
    {
        "순위": 3,
        "개선항목": "단위 정규화 및 검증",
        "난이도": "중",
        "예상효과": "오류율 -10-15%",
        "비용": "저",
        "시간": "3-5일"
    },
    {
        "순위": 4,
        "개선항목": "KOSIS 메타데이터 API 연동",
        "난이도": "고",
        "예상효과": "매칭 정확도 +30-50%",
        "비용": "중",
        "시간": "2-3주"
    },
    {
        "순위": 5,
        "개선항목": "수동 검증 세트 구축 (Human-in-the-loop)",
        "난이도": "중",
        "예상효과": "신뢰도 검증, 모니터링",
        "비용": "중",
        "시간": "1주 (지속)"
    },
])

display(priority_table)

print("\n💡 Tip: 1번과 2번을 먼저 구현하면 현재 기준선 대비 35-50% 개선이 가능합니다.")


# 8. HCX 성능 평가 (3단계)

확정 골드셋으로 후보 문장 분류, 기사 주장 탐지, 구조화·출처 성능과 오류를 평가합니다. 평가 도중 모델·프롬프트·파라미터를 바꾸지 않습니다.


In [ ]:
from datetime import datetime, timezone
from difflib import SequenceMatcher
from hashlib import sha256
import json, time, uuid
import pandas as pd
import requests

HCX_EVAL_CONFIG = {
    "model": "HCX-007", "prompt_version": "hcx-eval-v1.0",
    "temperature": 0.0, "topP": 0.8, "topK": 0, "maxTokens": 4096,
    "repeatPenalty": 1.0, "seed": 42, "timeout": 180,
    "retry_count": 0, "claim_match_threshold": 0.60,
}
HCX_EVAL_PROMPT = """한국어 뉴스에서 집계통계 주장을 판별하고 구조화하라. 개인 사례, 정책 목표·예산, 단일 시세, 단순 전망은 제외한다. 원문에 없는 값·시점·기관을 만들지 않는다. source_scope는 KOSIS계열, 비KOSIS공공, 민간, 해외, 없음 중 하나다."""
HCX_EVAL_SCHEMA = {"type":"object","properties":{
 "is_aggregate_claim":{"type":"boolean"}, "claim_class":{"type":"string"}, "source_scope":{"type":"string"},
 "claims":{"type":"array","items":{"type":"object","properties":{
  "claim_text":{"type":"string"},"evidence_quote":{"type":"string"},"indicator_raw":{"type":"string"},
  "population":{"type":"string"},"value":{"type":"array","items":{"type":"string"}},
  "unit":{"type":"array","items":{"type":"string"}},"change_type":{"type":"string"},
  "time_ref":{"type":"array","items":{"type":"string"}},"time_compare":{"type":"array","items":{"type":"string"}},
  "source_org_raw":{"type":"string"},"source_role":{"type":"string"},"source_evidence_quote":{"type":"string"}
 },"required":["claim_text","evidence_quote","value","unit","time_ref","source_org_raw"]}}
},"required":["is_aggregate_claim","claim_class","source_scope","claims"]}
EVAL_DIR=PROJECT_ROOT/'output'/'hcx_evaluation'; EVAL_DIR.mkdir(parents=True,exist_ok=True)
CANDIDATE_GOLD=PROJECT_ROOT/'data'/'gold'/'candidate_gold_final.csv'
ARTICLE_GOLD=PROJECT_ROOT/'data'/'gold'/'article_gold_final.csv'
CONFIG_HASH=sha256(json.dumps({"config":HCX_EVAL_CONFIG,"prompt":HCX_EVAL_PROMPT,"schema":HCX_EVAL_SCHEMA},ensure_ascii=False,sort_keys=True).encode()).hexdigest()
print('고정 평가 설정:',CONFIG_HASH)


## 8-1. HCX 고정 추론

모델명, 프롬프트 버전, 실행일, 파라미터, 요청 ID, 원본 응답, 파싱 결과, 지연시간과 오류 유형을 JSONL로 기록합니다. 파싱 실패는 실패 예측으로 보존하며 자동 재시도하지 않습니다.


In [ ]:
def run_hcx_eval(sample_id, text, input_type):
    request_id=str(uuid.uuid4()); started=time.perf_counter()
    record={"sample_id":str(sample_id),"input_type":input_type,"model":HCX_EVAL_CONFIG["model"],
      "prompt_version":HCX_EVAL_CONFIG["prompt_version"],"config_sha256":CONFIG_HASH,
      "executed_at":datetime.now(timezone.utc).isoformat(),"parameters":HCX_EVAL_CONFIG,
      "request_id":request_id,"raw_response":None,"parsed_result":None,"http_status":None,
      "latency_ms":None,"error_type":None,"error_message":None}
    headers={"Authorization":f"Bearer {NCP_API_KEY}","X-NCP-CLOVASTUDIO-REQUEST-ID":request_id,"Content-Type":"application/json"}
    body={"messages":[{"role":"system","content":HCX_EVAL_PROMPT},{"role":"user","content":f"입력 유형: {input_type}\n\n{text}"}],
      **{k:HCX_EVAL_CONFIG[k] for k in ["temperature","topP","topK","maxTokens","repeatPenalty","seed"]},
      "responseFormat":{"type":"json","schema":HCX_EVAL_SCHEMA}}
    try:
        response=requests.post(NCP_API_URL,headers=headers,json=body,timeout=HCX_EVAL_CONFIG["timeout"])
        record["http_status"]=response.status_code; response.raise_for_status()
        record["raw_response"]=response.json().get("result",{}).get("message",{}).get("content")
        record["parsed_result"]=json.loads(record["raw_response"])
    except requests.Timeout as e: record.update(error_type="timeout",error_message=str(e))
    except requests.HTTPError as e: record.update(error_type="http_error",error_message=str(e))
    except json.JSONDecodeError as e: record.update(error_type="json_parse_failure",error_message=str(e))
    except Exception as e: record.update(error_type=type(e).__name__,error_message=str(e))
    record["latency_ms"]=round((time.perf_counter()-started)*1000,2); return record

def run_eval_batch(gold_path,input_type,output_path,limit=None):
    gold=pd.read_csv(gold_path,encoding='utf-8-sig')
    if gold.sample_id.duplicated().any(): raise ValueError('sample_id 중복')
    choices=['text','sentence'] if input_type=='candidate' else ['본문_정제','article_text','text']
    text_col=next((c for c in choices if c in gold),None)
    if not text_col: raise KeyError(f'본문 컬럼 필요: {choices}')
    rows=[]
    for _,r in (gold if limit is None else gold.head(limit)).iterrows():
        result=run_hcx_eval(r.sample_id,str(r[text_col]),input_type); rows.append(result)
        with output_path.open('a',encoding='utf-8') as f: f.write(json.dumps(result,ensure_ascii=False)+'\n')
    return pd.DataFrame(rows)

EVALUATION_LIMIT=1  # 공식 전체 평가는 None
# candidate_runs=run_eval_batch(CANDIDATE_GOLD,'candidate',EVAL_DIR/'hcx_candidate_runs.jsonl',EVALUATION_LIMIT)
# article_runs=run_eval_batch(ARTICLE_GOLD,'article',EVAL_DIR/'hcx_article_runs.jsonl',EVALUATION_LIMIT)


## 8-2. 후보 문장 분류 평가

Precision, Recall, F1, Accuracy, FPR/FNR, claim_class별 지표와 confusion matrix를 계산합니다. `both/relaxed_only`, 단일/다중 값, change_type, 출처 명시 여부, 검색 레이블별로 분리합니다.


In [ ]:
def binary_metrics(y,p):
    pairs=list(zip(map(bool,y),map(bool,p))); tp=sum(a and b for a,b in pairs); tn=sum(not a and not b for a,b in pairs); fp=sum(not a and b for a,b in pairs); fn=sum(a and not b for a,b in pairs)
    div=lambda a,b:a/b if b else 0; pr=div(tp,tp+fp); rc=div(tp,tp+fn)
    return {"n":len(pairs),"tp":tp,"tn":tn,"fp":fp,"fn":fn,"precision":pr,"recall":rc,"f1":div(2*pr*rc,pr+rc),"accuracy":div(tp+tn,len(pairs)),"fpr":div(fp,fp+tn),"fnr":div(fn,fn+tp)}

def class_metrics(y,p):
    labels=sorted(set(map(str,y))|set(map(str,p))); matrix={a:{b:0 for b in labels} for a in labels}
    for a,b in zip(map(str,y),map(str,p)): matrix[a][b]+=1
    return {"per_class":{x:binary_metrics([str(v)==x for v in y],[str(v)==x for v in p]) for x in labels},"confusion_matrix":matrix}

def load_predictions(path):
    rows=[]
    for line in path.read_text(encoding='utf-8').splitlines():
        r=json.loads(line); x=r.get('parsed_result') or {}; rows.append({**r,"pred_is_aggregate_claim":x.get('is_aggregate_claim',False),"pred_claim_class":x.get('claim_class','parse_failure'),"pred_source_scope":x.get('source_scope','parse_failure'),"pred_claims_json":json.dumps(x.get('claims',[]),ensure_ascii=False)})
    return pd.DataFrame(rows)

def evaluate_candidates():
    g=pd.read_csv(CANDIDATE_GOLD,encoding='utf-8-sig'); d=g.merge(load_predictions(EVAL_DIR/'hcx_candidate_runs.jsonl'),on='sample_id',validate='one_to_one')
    truth=d.gold_is_aggregate_claim.astype(str).str.lower().isin(['true','1','yes']); result={"overall":binary_metrics(truth,d.pred_is_aggregate_claim),"claim_class":class_metrics(d.gold_claim_class,d.pred_claim_class),"strata":{}}
    for key,names in {"filter_group":["filter_group","candidate_group"],"value_count":["value_cardinality","value_count_group"],"change_type":["change_type","gold_change_type"],"source":["source_explicit","has_source"],"search_label":["search_label","검색 레이블"]}.items():
        col=next((x for x in names if x in d),None)
        if col: result['strata'][key]={str(v):binary_metrics(truth.loc[x.index],x.pred_is_aggregate_claim) for v,x in d.groupby(col,dropna=False)}
    d.to_csv(EVAL_DIR/'hcx_candidate_predictions.csv',index=False,encoding='utf-8-sig'); (EVAL_DIR/'hcx_candidate_metrics.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8'); return d,result
# candidate_predictions,candidate_metrics=evaluate_candidates()


## 8-3. 기사 주장 1:1 매칭·구조화·출처·오류 평가

원문 근거, 핵심 지표, 값, 단위, 시점을 함께 비교합니다. 임계값 주변 사례는 사람 검토 대상으로 표시합니다. 정확히 매칭된 주장에 한해 구조 필드를 평가합니다.


In [ ]:
def as_list(x): return [str(v).strip().lower().replace(',','') for v in (x if isinstance(x,list) else ([] if x in [None,''] else [x]))]
def overlap(a,b):
    a,b=set(as_list(a)),set(as_list(b)); return len(a&b)/len(a|b) if a|b else 1
def match_score(g,p):
    text=SequenceMatcher(None,str(g.get('claim_text','')),str(p.get('claim_text',''))).ratio()
    indicator=SequenceMatcher(None,str(g.get('indicator_raw','')),str(p.get('indicator_raw',''))).ratio()
    return .4*text+.2*overlap(g.get('value'),p.get('value'))+.1*overlap(g.get('unit'),p.get('unit'))+.15*overlap(g.get('time_ref'),p.get('time_ref'))+.15*indicator
def match_claims(gs,ps):
    threshold=HCX_EVAL_CONFIG['claim_match_threshold']; candidates=sorted([(match_score(g,p),i,j) for i,g in enumerate(gs) for j,p in enumerate(ps)],reverse=True); ug=set(); up=set(); rows=[]
    for score,i,j in candidates:
        if score>=threshold and i not in ug and j not in up: ug.add(i);up.add(j);rows.append({"gold_index":i,"pred_index":j,"status":"matched","score":score,"manual_review_required":abs(score-threshold)<=.05})
    rows += [{"gold_index":i,"pred_index":None,"status":"missed","score":0} for i in range(len(gs)) if i not in ug]
    rows += [{"gold_index":None,"pred_index":j,"status":"extra","score":0} for j in range(len(ps)) if j not in up]; return rows

def evaluate_articles():
    g=pd.read_csv(ARTICLE_GOLD,encoding='utf-8-sig'); d=g.merge(load_predictions(EVAL_DIR/'hcx_article_runs.jsonl'),on='sample_id',validate='one_to_one'); matches=[]; summaries=[]
    for _,r in d.iterrows():
        gs=json.loads(r.gold_claims_json); ps=json.loads(r.pred_claims_json); ms=match_claims(gs,ps); tp=sum(x['status']=='matched' for x in ms)
        for x in ms: matches.append({"sample_id":r.sample_id,**x,"gold_claim_json":json.dumps(gs[x['gold_index']],ensure_ascii=False) if x.get('gold_index') is not None else None,"pred_claim_json":json.dumps(ps[x['pred_index']],ensure_ascii=False) if x.get('pred_index') is not None else None})
        summaries.append({"sample_id":r.sample_id,"gold_count":len(gs),"pred_count":len(ps),"matched":tp,"complete":tp==len(gs)==len(ps),"missed":tp<len(gs),"extra":tp<len(ps),"no_claim_fp":len(gs)==0<len(ps),"latency_ms":r.latency_ms,"error_type":r.error_type})
    m=pd.DataFrame(matches); a=pd.DataFrame(summaries); tp=a.matched.sum(); gn=a.gold_count.sum(); pn=a.pred_count.sum(); pr=tp/pn if pn else 0; rc=tp/gn if gn else 0
    metrics={"precision":pr,"recall":rc,"f1":2*pr*rc/(pr+rc) if pr+rc else 0,"complete_article_rate":a.complete.mean(),"missing_article_rate":a.missed.mean(),"over_extraction_rate":a.extra.mean(),"no_claim_false_positive_rate":a.loc[a.gold_count.eq(0),'no_claim_fp'].mean(),"parse_failure_rate":a.error_type.eq('json_parse_failure').mean(),"mean_latency_ms":a.latency_ms.mean()}
    d.to_csv(EVAL_DIR/'hcx_article_predictions.csv',index=False,encoding='utf-8-sig');m.to_csv(EVAL_DIR/'hcx_claim_matching.csv',index=False,encoding='utf-8-sig');(EVAL_DIR/'hcx_article_metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8');return d,m,a,metrics

def evaluate_structure_source(m):
    rows=[]
    for _,r in m[m.status.eq('matched')].iterrows():
        g=json.loads(r.gold_claim_json);p=json.loads(r.pred_claim_json); z={"sample_id":r.sample_id}
        for f in ['value','unit','time_ref','time_compare','indicator_raw','source_org_raw']: z[f+'_correct']=as_list(g.get(f))==as_list(p.get(f))
        z['value_count_correct']=len(as_list(g.get('value')))==len(as_list(p.get('value')));z['time_count_correct']=len(as_list(g.get('time_ref')))==len(as_list(p.get('time_ref')));z['order_correct']=list(zip(as_list(g.get('value')),as_list(g.get('time_ref'))))==list(zip(as_list(p.get('value')),as_list(p.get('time_ref'))));z['gold_source']=bool(as_list(g.get('source_org_raw')));z['pred_source']=bool(as_list(p.get('source_org_raw')));rows.append(z)
    d=pd.DataFrame(rows); metrics={"field_accuracy":{c:float(d[c].mean()) for c in d if c.endswith('_correct')},"source_recall":d.loc[d.gold_source,'pred_source'].mean() if len(d) and d.gold_source.any() else None,"source_hallucination_rate":d.loc[~d.gold_source,'pred_source'].mean() if len(d) and (~d.gold_source).any() else None}
    d.to_csv(EVAL_DIR/'hcx_structure_source_details.csv',index=False,encoding='utf-8-sig');(EVAL_DIR/'hcx_structure_source_metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8');return d,metrics
# article_predictions,claim_matching,article_summary,article_metrics=evaluate_articles()
# structure_details,structure_metrics=evaluate_structure_source(claim_matching)


## 8-4. 오류 분석·최종 적용 보고서

누락, 과다 추출, 값·단위·시점·나열 순서·출처·JSON 오류를 집계하고 대표 사례와 수정 방향을 사람이 보완합니다. 동일 골드셋의 규칙 기반 및 규칙+HCX 결과를 비교해 비용·지연시간 대비 적용 여부를 결정합니다.


In [ ]:
def build_final_report(candidate_metrics,article_metrics,structure_metrics,claim_matching):
    errors=claim_matching[~claim_matching.status.eq('matched')].copy(); errors['error_type']=errors.status.map({'missed':'주장·값·출처 누락','extra':'오탐·출처 환각'}); errors.to_csv(EVAL_DIR/'hcx_error_cases.csv',index=False,encoding='utf-8-sig')
    baseline_path=EVAL_DIR/'baseline_metrics.json'; baseline=json.loads(baseline_path.read_text(encoding='utf-8')) if baseline_path.is_file() else {'status':'규칙 기반/규칙+HCX 동일 골드셋 결과 입력 필요'}
    c=candidate_metrics['overall']; lines=['# HCX 성능 평가 보고서','',f'- 모델/프롬프트: {HCX_EVAL_CONFIG["model"]} / {HCX_EVAL_CONFIG["prompt_version"]}',f'- 설정 해시: `{CONFIG_HASH}`','', '## 후보 성능',json.dumps(c,ensure_ascii=False,indent=2),'','## 기사 주장 성능',json.dumps(article_metrics,ensure_ascii=False,indent=2),'','## 구조·출처 성능',json.dumps(structure_metrics,ensure_ascii=False,indent=2),'','## 규칙 기반 / HCX / 결합 비교',json.dumps(baseline,ensure_ascii=False,indent=2),'','## 적용 판단','- both 보존율과 relaxed_only 오탐 제거율을 층별 지표로 확인','- Precision·Recall 개선이 비용과 기사당 지연시간을 감수할 수준인지 확인','- 파싱 실패와 출처 환각이 허용 가능한지 확인','- 오류별로 프롬프트, 코드 후처리, 추가 라벨링 중 조치 결정']
    path=EVAL_DIR/'hcx_evaluation_report.md';path.write_text('\n'.join(lines),encoding='utf-8');return path
# report_path=build_final_report(candidate_metrics,article_metrics,structure_metrics,claim_matching)


## 6-D. 6-A 주장 기반 KOSIS 기관·통계표 프롬프트 매핑

6-A의 주장 추출 코드와 결과를 수정하지 않고 후속 단계로 사용합니다. 100MB가 넘는 통계표 트리를 한 번에 모델에 전달하지 않고 다음 두 번의 프롬프트 호출로 나눕니다.

1. 주장 + 실제 KOSIS 기관/분야 목록 → 관련 기관·분야 선택
2. 선택 범위의 실제 통계표 후보 → 최종 기관·통계표 선택
3. 선택된 `org_id`, `tbl_id`로 KOSIS 통계자료 API 조회

모델은 제공된 ID만 고를 수 있으며, 후보에 없는 기관이나 표를 생성하지 못하도록 JSON Schema와 후처리 검증을 함께 적용합니다.


### 6-D-1. 경로, 카탈로그 및 프롬프트용 선택지 구성

로컬 프로젝트와 질문에 명시된 Google Drive 경로를 모두 지원합니다. `kosis_table_tree.json`은 실제 KOSIS API로 수집한 통계표 목록이며, 각 표의 기관·분야·표 ID를 프롬프트 선택지로 사용합니다.


In [ ]:
KOSIS_ORG_PATH_6D = locate_file_6c("kosis_org_names.json") if "locate_file_6c" in globals() else None
KOSIS_TREE_PATH_6D = locate_file_6c("kosis_table_tree.json") if "locate_file_6c" in globals() else None

if KOSIS_ORG_PATH_6D is None or KOSIS_TREE_PATH_6D is None:
    def locate_kosis_file_6d(filename):
        candidates = [
            Path.cwd() / "data" / filename,
            Path.cwd() / filename,
            Path("/content/drive/MyDrive/news_verification/data/통계표") / filename,
            Path("/content/drive/MyDrive/news_verification/data") / filename,
        ]
        for candidate in candidates:
            if candidate.is_file():
                return candidate
        raise FileNotFoundError(f"{filename}을 찾지 못했습니다:\n" + "\n".join(map(str, candidates)))
    KOSIS_ORG_PATH_6D = locate_kosis_file_6d("kosis_org_names.json")
    KOSIS_TREE_PATH_6D = locate_kosis_file_6d("kosis_table_tree.json")

kosis_org_names_6d = json.loads(KOSIS_ORG_PATH_6D.read_text(encoding="utf-8"))
kosis_tree_6d = json.loads(KOSIS_TREE_PATH_6D.read_text(encoding="utf-8"))

def build_kosis_catalog_6d(tree, org_names):
    rows = []
    roots = tree.values() if isinstance(tree, dict) else tree
    for root in roots:
        for leaf in root.get("leaves", []):
            path = [str(value) for value in leaf.get("path", []) if str(value).strip()]
            org_id = str(leaf.get("org_id", ""))
            rows.append({
                "org_id": org_id,
                "org_name": str(org_names.get(org_id, "")),
                "tbl_id": str(leaf.get("tbl_id", "")),
                "tbl_nm": str(leaf.get("tbl_nm", "")),
                "stat_id": str(leaf.get("stat_id", "")),
                "top_category": path[0] if path else str(root.get("top_nm", "")),
                "data_type": path[1] if len(path) > 1 else (path[0] if path else ""),
                "category_path": " > ".join(path),
            })
    return pd.DataFrame(rows).drop_duplicates(["org_id", "tbl_id"]).reset_index(drop=True)

kosis_catalog_6d = build_kosis_catalog_6d(kosis_tree_6d, kosis_org_names_6d)
kosis_scope_6d = (
    kosis_catalog_6d.groupby(["org_id", "org_name", "top_category", "data_type"], dropna=False)
    .size().reset_index(name="table_count")
)
print(f"기관 {kosis_catalog_6d['org_id'].nunique():,}개 / 통계표 {len(kosis_catalog_6d):,}개 / 기관·분야 선택지 {len(kosis_scope_6d):,}개")
display(kosis_scope_6d.head())


### 6-D-2. 공통 HCX JSON 호출과 1단계 기관·데이터 종류 선택

1단계 출력의 `org_id`, `top_category`, `data_type`은 입력 선택지에 실제로 존재하는 값만 허용합니다. `kosis_usable=false`는 해당 주장이 정기 집계 통계가 아니거나 KOSIS 범위 밖이라고 판단한 경우입니다.


In [ ]:
KOSIS_SCOPE_SCHEMA_6D = {
    "type": "object",
    "properties": {
        "kosis_usable": {"type": "boolean"},
        "reason": {"type": "string"},
        "search_keywords": {"type": "array", "items": {"type": "string"}},
        "scopes": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "org_id": {"type": "string"}, "org_name": {"type": "string"},
                "top_category": {"type": "string"}, "data_type": {"type": "string"},
                "reason": {"type": "string"}
            },
            "required": ["org_id", "org_name", "top_category", "data_type", "reason"]
        }}
    },
    "required": ["kosis_usable", "reason", "search_keywords", "scopes"]
}

def call_hcx_json_6d(system_prompt, user_payload, schema, max_retries=3):
    if not NCP_API_KEY:
        raise RuntimeError("NCP_CLOVASTUDIO_API_KEY가 설정되지 않았습니다.")
    headers = {
        "Authorization": f"Bearer {NCP_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json", "Accept": "application/json",
    }
    body = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
        ],
        "topP": 0.8, "topK": 0, "temperature": 0.1,
        "repetitionPenalty": 1.1, "maxCompletionTokens": 4096,
        "thinking": {"effort": "none"},
        "responseFormat": {"type": "json", "schema": schema},
    }
    for attempt in range(max_retries):
        response = requests.post(NCP_API_URL, headers=headers, json=body, timeout=180)
        if response.status_code == 429 or response.status_code >= 500:
            if attempt + 1 < max_retries:
                time.sleep(2 ** attempt)
                continue
        response.raise_for_status()
        payload = response.json()
        result = payload.get("result", payload)
        content = result.get("message", {}).get("content", "")
        if not content:
            raise RuntimeError(f"HCX 응답에 content가 없습니다: {payload}")
        return json.loads(content), result.get("usage", {})
    raise RuntimeError("HCX API 재시도 횟수를 초과했습니다.")

SYSTEM_SCOPE_PROMPT_6D = """당신은 뉴스 주장을 KOSIS 공식 통계에 연결하는 검색 설계자다.
입력 claim을 검증할 가능성이 있는 기관과 데이터 종류를 반드시 scope_options에서만 최대 5개 고른다.
기관명이 기사에 명시돼도 실제 작성기관과 다를 수 있으므로 지표 의미를 함께 고려한다.
정기적으로 집계되는 인구·가구·고용·물가·산업·보건·교육 등의 통계만 kosis_usable=true로 판단한다.
기업 내부자료, 일회성 사건 집계, 전망·목표, 해외기관 자료는 적절한 선택지가 없으면 false로 판단한다.
search_keywords에는 표 이름 검색에 유용한 핵심 명사와 동의어만 넣는다.
입력에 없는 ID나 이름을 만들지 말고 JSON 외 설명을 출력하지 않는다."""

def select_kosis_scopes_6d(claim, max_options=2500):
    options = kosis_scope_6d.sort_values("table_count", ascending=False).head(max_options).to_dict("records")
    parsed, usage = call_hcx_json_6d(
        SYSTEM_SCOPE_PROMPT_6D,
        {"claim": claim, "scope_options": options},
        KOSIS_SCOPE_SCHEMA_6D,
    )
    valid = set(map(tuple, kosis_scope_6d[["org_id", "top_category", "data_type"]].astype(str).to_numpy()))
    parsed["scopes"] = [
        item for item in parsed.get("scopes", [])
        if (str(item.get("org_id", "")), str(item.get("top_category", "")), str(item.get("data_type", ""))) in valid
    ][:5]
    if not parsed["scopes"]:
        parsed["kosis_usable"] = False
    return parsed, usage


### 6-D-3. 2단계 통계표 선택

선택된 기관·분야 안에서 키워드 점수로 프롬프트 크기만 줄인 뒤, HCX가 주장에 필요한 시점·지역·대상·단위를 고려해 최종 표를 고릅니다. 문자열 점수는 최종 판단이 아니라 LLM에 제공할 후보 축소에만 사용됩니다.


In [ ]:
KOSIS_TABLE_SCHEMA_6D = {
    "type": "object",
    "properties": {
        "matches": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "rank": {"type": "integer"}, "org_id": {"type": "string"},
                "org_name": {"type": "string"}, "tbl_id": {"type": "string"},
                "tbl_nm": {"type": "string"}, "stat_id": {"type": "string"},
                "category_path": {"type": "string"}, "reason": {"type": "string"},
                "query_plan": {"type": "string"}, "confidence": {"type": "number"}
            },
            "required": ["rank", "org_id", "org_name", "tbl_id", "tbl_nm", "stat_id", "category_path", "reason", "query_plan", "confidence"]
        }},
        "no_match_reason": {"type": "string"}
    },
    "required": ["matches", "no_match_reason"]
}

TOKEN_PATTERN_6D = re.compile(r"[가-힣A-Za-z0-9]{2,}")

def shortlist_tables_6d(scope_result, limit=120):
    scopes = scope_result.get("scopes", [])
    if not scopes:
        return kosis_catalog_6d.iloc[:0].copy()
    mask = pd.Series(False, index=kosis_catalog_6d.index)
    for scope in scopes:
        mask |= (
            (kosis_catalog_6d["org_id"] == str(scope["org_id"]))
            & (kosis_catalog_6d["top_category"] == str(scope["top_category"]))
            & (kosis_catalog_6d["data_type"] == str(scope["data_type"]))
        )
    candidates = kosis_catalog_6d[mask].copy()
    keywords = [str(word).casefold() for word in scope_result.get("search_keywords", []) if len(str(word)) >= 2]
    document = (candidates["tbl_nm"] + " " + candidates["category_path"] + " " + candidates["org_name"]).str.casefold()
    candidates["keyword_score"] = 0
    for keyword in keywords:
        candidates["keyword_score"] += document.str.contains(re.escape(keyword), regex=True, na=False).astype(int)
    return candidates.sort_values(["keyword_score", "tbl_nm"], ascending=[False, True]).head(limit)

SYSTEM_TABLE_PROMPT_6D = """당신은 KOSIS 통계표 검색 전문가다.
claim 검증에 실제로 사용할 수 있는 표를 table_options에서만 최대 5개 선택한다.
지표 의미, 모집단, 지역 범위, 시점·주기, 값의 단위를 함께 고려한다.
표 이름만으로 세부 항목 존재를 확정할 수 없으면 query_plan에 API 메타/자료 조회 후 확인할 항목을 명시한다.
후보가 모두 부적절하면 matches를 빈 배열로 반환한다.
입력에 없는 기관명·표명·ID를 만들거나 수정하지 말고 JSON 외 설명을 출력하지 않는다."""

def select_kosis_tables_6d(claim, scope_result, shortlist_limit=120):
    candidates = shortlist_tables_6d(scope_result, limit=shortlist_limit)
    if candidates.empty:
        return {"matches": [], "no_match_reason": scope_result.get("reason", "적합한 기관·분야 없음")}, {}, candidates
    option_columns = ["org_id", "org_name", "tbl_id", "tbl_nm", "stat_id", "category_path"]
    parsed, usage = call_hcx_json_6d(
        SYSTEM_TABLE_PROMPT_6D,
        {"claim": claim, "table_options": candidates[option_columns].to_dict("records")},
        KOSIS_TABLE_SCHEMA_6D,
    )
    canonical = {
        (str(row["org_id"]), str(row["tbl_id"])): row[option_columns[1:]].to_dict()
        for _, row in candidates.iterrows()
    }
    valid_matches = []
    for item in parsed.get("matches", []):
        key = (str(item.get("org_id", "")), str(item.get("tbl_id", "")))
        if key not in canonical:
            continue
        source = canonical[key]
        item.update({"org_id": key[0], "tbl_id": key[1], **source})
        valid_matches.append(item)
    parsed["matches"] = valid_matches[:5]
    return parsed, usage, candidates

def map_claim_to_kosis_6d(claim):
    scope_result, scope_usage = select_kosis_scopes_6d(claim)
    table_result, table_usage, candidates = select_kosis_tables_6d(claim, scope_result)
    usage = {key: int(scope_usage.get(key, 0) or 0) + int(table_usage.get(key, 0) or 0)
             for key in {**scope_usage, **table_usage}}
    return {"claim": claim, "scope_result": scope_result, "table_result": table_result,
            "shortlist_count": len(candidates), "usage": usage}


### 6-D-4. 6-A 결과로 테스트

6-A의 단일 기사 결과 `test_claims`를 우선 사용하고, 없으면 배치 결과 `claims_df`의 첫 행을 사용합니다. API 비용 방지를 위해 기본 실행 스위치는 꺼져 있습니다.


In [ ]:
RUN_KOSIS_MAPPING_TEST_6D = False

def first_claim_from_6a_6d():
    if "test_claims" in globals() and test_claims:
        return test_claims[0]
    if "claims_df" in globals() and not claims_df.empty:
        row = claims_df.iloc[0].dropna().to_dict()
        raw = row.get("numeric_values_json", "[]")
        row["numeric_values"] = json.loads(raw) if isinstance(raw, str) else []
        return row
    raise RuntimeError("먼저 6-A 테스트 또는 배치 추출 셀을 실행하세요.")

if RUN_KOSIS_MAPPING_TEST_6D:
    kosis_mapping_test_6d = map_claim_to_kosis_6d(first_claim_from_6a_6d())
    print("선택 기관·분야")
    display(pd.DataFrame(kosis_mapping_test_6d["scope_result"]["scopes"]))
    print("선택 통계표")
    display(pd.DataFrame(kosis_mapping_test_6d["table_result"]["matches"]))


### 6-D-5. 선택한 통계표의 KOSIS API 자료 조회

`.env`의 `KOSIS_API_KEY`를 사용합니다. 기간을 지정하지 않으면 최근 10개 시점을 요청하며, KOSIS 응답이 오류 객체이면 메시지와 함께 예외를 발생시킵니다.


In [ ]:
KOSIS_DATA_API_URL_6D = "https://kosis.kr/openapi/Param/statisticsParameterData.do"
KOSIS_API_KEY_6D = os.getenv("KOSIS_API_KEY", "").strip()

def fetch_kosis_data_6d(org_id, tbl_id, start_period="", end_period="", period="Y", recent_count=10):
    if not KOSIS_API_KEY_6D:
        raise RuntimeError(".env에 KOSIS_API_KEY를 설정하세요.")
    params = {
        "method": "getList", "apiKey": KOSIS_API_KEY_6D,
        "orgId": str(org_id), "tblId": str(tbl_id),
        "itmId": "ALL", "objL1": "ALL", "objL2": "ALL", "objL3": "ALL",
        "objL4": "ALL", "objL5": "ALL", "objL6": "ALL", "objL7": "ALL", "objL8": "ALL",
        "format": "json", "jsonVD": "Y", "prdSe": period,
    }
    if start_period and end_period:
        params.update({"startPrdDe": str(start_period), "endPrdDe": str(end_period)})
    else:
        params["newEstPrdCnt"] = str(recent_count)
    response = requests.get(KOSIS_DATA_API_URL_6D, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict):
        message = payload.get("errMsg") or payload.get("message") or str(payload)
        raise RuntimeError(f"KOSIS API 오류: {message}")
    return pd.DataFrame(payload)

RUN_KOSIS_DATA_TEST_6D = False
if RUN_KOSIS_DATA_TEST_6D:
    matches = kosis_mapping_test_6d["table_result"]["matches"]
    if not matches:
        print("조회할 통계표 후보가 없습니다.")
    else:
        best = matches[0]
        kosis_raw_data_6d = fetch_kosis_data_6d(best["org_id"], best["tbl_id"])
        print(best["org_name"], "/", best["tbl_nm"], "/", len(kosis_raw_data_6d), "행")
        display(kosis_raw_data_6d.head(20))


### 6-D-6. 배치 매핑 및 체크포인트

6-A 배치 CSV의 각 주장을 독립적으로 매핑합니다. 성공한 주장 번호는 재실행에서 건너뛰며, 기관·통계표 후보와 판단 이유를 JSONL에 즉시 저장합니다.


In [ ]:
RUN_KOSIS_MAPPING_BATCH_6D = False
MAX_CLAIMS_6D = 10  # 전체는 None
KOSIS_MAPPING_CHECKPOINT_6D = OUTPUT_DIR_6C / f"{OUTPUT_STEM_6C}_6D_kosis_mapping.jsonl" if "OUTPUT_DIR_6C" in globals() else Path.cwd() / "kosis_mapping_6d.jsonl"
KOSIS_MAPPING_OUTPUT_6D = KOSIS_MAPPING_CHECKPOINT_6D.with_suffix(".csv")

def run_kosis_mapping_batch_6d(frame, max_claims=MAX_CLAIMS_6D):
    completed = set()
    if KOSIS_MAPPING_CHECKPOINT_6D.is_file():
        for line in KOSIS_MAPPING_CHECKPOINT_6D.read_text(encoding="utf-8").splitlines():
            try:
                record = json.loads(line)
                if record.get("status") == "success":
                    completed.add((str(record.get("URL", "")), str(record.get("claim_number", ""))))
            except json.JSONDecodeError:
                pass
    targets = frame.copy()
    targets = targets[~targets.apply(lambda row: (str(row.get("URL", "")), str(row.get("claim_number", ""))) in completed, axis=1)]
    if max_claims is not None:
        targets = targets.head(max_claims)
    with KOSIS_MAPPING_CHECKPOINT_6D.open("a", encoding="utf-8") as checkpoint:
        for sequence, (_, row) in enumerate(targets.iterrows(), 1):
            record = {"URL": str(row.get("URL", "")), "claim_number": str(row.get("claim_number", "")),
                      "claim_text": str(row.get("claim_text", ""))}
            try:
                claim = row.dropna().to_dict()
                raw_values = claim.get("numeric_values_json", "[]")
                claim["numeric_values"] = json.loads(raw_values) if isinstance(raw_values, str) else []
                result = map_claim_to_kosis_6d(claim)
                record.update({"status": "success", **result})
            except Exception as error:
                record.update({"status": "failed", "error_type": type(error).__name__, "error": str(error)})
            checkpoint.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            checkpoint.flush()
            print(f"[{sequence}/{len(targets)}] {record['status']} - {record['claim_text'][:60]}")
            time.sleep(REQUEST_INTERVAL_SECONDS)

    flat = []
    for line in KOSIS_MAPPING_CHECKPOINT_6D.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        if record.get("status") != "success":
            continue
        base = {"URL": record["URL"], "claim_number": record["claim_number"], "claim_text": record["claim_text"],
                "kosis_usable": record["scope_result"].get("kosis_usable"),
                "scope_reason": record["scope_result"].get("reason", "")}
        matches = record["table_result"].get("matches", [])
        if not matches:
            flat.append({**base, "rank": None, "no_match_reason": record["table_result"].get("no_match_reason", "")})
        else:
            flat.extend({**base, **match} for match in matches)
    result_frame = pd.DataFrame(flat)
    result_frame.to_csv(KOSIS_MAPPING_OUTPUT_6D, index=False, encoding="utf-8-sig")
    return result_frame

if RUN_KOSIS_MAPPING_BATCH_6D:
    if "claims_df" not in globals() or claims_df.empty:
        raise RuntimeError("6-A 배치 추출 셀을 먼저 실행하세요.")
    kosis_mapping_results_6d = run_kosis_mapping_batch_6d(claims_df)
    display(kosis_mapping_results_6d.head(20))
